In [1]:
# Cell 0 | 环境、路径与输入文件审计

from pathlib import Path
from datetime import datetime
import hashlib
import importlib.metadata
import json
import os
import platform
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from rdkit import Chem, rdBase

warnings.filterwarnings("ignore", category=FutureWarning)

RUN_CODE_VERSION = "01_exact_CO2_pair_match_diagnostic_v1_20260805"

# =========================
# 1. 固定输入路径
# =========================

PROJECT_ROOT = Path(r"D:\jupyter\IL_ML_Project")

CANDIDATE_PATH = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
    / "outputs"
    / "virtual_IL_lightgbm_viscosity"
    / "tables"
    / "virtual_IL_low_viscosity_ranking_nonhalide.csv"
)

REASONABLE_LIBRARY_PATH = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
    / "outputs"
    / "virtual_IL_lightgbm_viscosity"
    / "tables"
    / "virtual_IL_library_after_reasonability_filter.csv"
)

SOURCE_CATION_PATH = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
    / "outputs"
    / "virtual_IL_lightgbm_viscosity"
    / "tables"
    / "source_cation_frequency.csv"
)

SOURCE_ANION_PATH = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
    / "outputs"
    / "virtual_IL_lightgbm_viscosity"
    / "tables"
    / "source_anion_frequency.csv"
)

GENERATION_AUDIT_PATH = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
    / "outputs"
    / "virtual_IL_lightgbm_viscosity"
    / "logs"
    / "virtual_IL_generation_audit.json"
)

REASONABILITY_AUDIT_PATH = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
    / "outputs"
    / "virtual_IL_lightgbm_viscosity"
    / "logs"
    / "virtual_IL_reasonability_filter_audit.json"
)

PREDICTION_MANIFEST_PATH = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
    / "outputs"
    / "virtual_IL_lightgbm_viscosity"
    / "logs"
    / "virtual_IL_frozen_LightGBM_prediction_manifest.json"
)

RUN_MANIFEST_PATH = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
    / "outputs"
    / "virtual_IL_lightgbm_viscosity"
    / "virtual_viscosity_run_manifest.json"
)

CO2_CSV_PATH = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
    / "external_data"
    / "co2_solubility"
    / "co2_solubility.csv"
)

CO2_SOURCE_XLS_PATH = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
    / "external_data"
    / "co2_solubility"
    / "co2_solubility_source_database.xls"
)

# =========================
# 2. 固定输出目录
# =========================

OUT_DIR = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
    / "outputs"
    / "low_viscosity_CO2_solubility_crosscheck"
)

TABLE_DIR = OUT_DIR / "tables"
LOG_DIR = OUT_DIR / "logs"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATHS = {
    "environment_input_audit": (
        LOG_DIR / "environment_input_audit.json"
    ),
    "input_dataset_audit": (
        TABLE_DIR / "input_dataset_audit.csv"
    ),
    "canonicalization_audit": (
        TABLE_DIR / "co2_database_smiles_canonicalization_audit.csv"
    ),
    "invalid_smiles_records": (
        TABLE_DIR / "co2_database_invalid_smiles_records.csv"
    ),
    "excluded_records": (
        TABLE_DIR
        / "co2_database_records_excluded_from_exact_matching.csv"
    ),
    "candidate_canonical_keys": (
        TABLE_DIR / "candidate_pair_keys_canonical.csv"
    ),
    "co2_canonicalized_records": (
        TABLE_DIR / "co2_database_canonicalized_for_matching.csv"
    ),
    "exact_pair_match_summary": (
        TABLE_DIR / "candidate_CO2_exact_pair_match_summary.csv"
    ),
    "exact_pair_matched_records": (
        TABLE_DIR / "candidate_CO2_exact_pair_matched_records.csv"
    ),
    "exact_match_counts_by_scope": (
        TABLE_DIR / "candidate_CO2_exact_match_counts_by_scope.csv"
    ),
    "temperature_coverage": (
        TABLE_DIR / "exact_match_temperature_coverage.csv"
    ),
    "pressure_coverage": (
        TABLE_DIR
        / "exact_match_pressure_coverage_by_temperature.csv"
    ),
    "candidate_condition_summary": (
        TABLE_DIR / "exact_match_candidate_condition_summary.csv"
    ),
    "records_T29815_window": (
        TABLE_DIR / "exact_match_records_T29815_window.csv"
    ),
    "diagnostic_manifest": (
        LOG_DIR / "CO2_exact_match_diagnostic_manifest.json"
    ),
}

REFERENCE_T_K = 298.15
REFERENCE_T_TOL_K = 0.50
CO2_REFERENCE_SMILES = "O=C=O"

# =========================
# 3. 通用保存与审计函数
# =========================

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as file_obj:
        while True:
            chunk = file_obj.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def save_csv_atomic(
    dataframe: pd.DataFrame,
    path: Path,
) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_name(path.name + ".tmp")

    dataframe.to_csv(
        temporary_path,
        index=False,
        encoding="utf-8-sig",
    )

    os.replace(
        temporary_path,
        path,
    )


def save_json_atomic(
    data: dict,
    path: Path,
) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_name(path.name + ".tmp")

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file_obj:
        json.dump(
            data,
            file_obj,
            ensure_ascii=False,
            indent=2,
        )

    os.replace(
        temporary_path,
        path,
    )


def read_json(path: Path) -> dict:
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as file_obj:
        return json.load(file_obj)


def get_package_version(package_name: str) -> str:
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return "not_installed"


# =========================
# 4. 检查全部必要输入
# =========================

REQUIRED_INPUTS = [
    CANDIDATE_PATH,
    REASONABLE_LIBRARY_PATH,
    SOURCE_CATION_PATH,
    SOURCE_ANION_PATH,
    GENERATION_AUDIT_PATH,
    REASONABILITY_AUDIT_PATH,
    PREDICTION_MANIFEST_PATH,
    RUN_MANIFEST_PATH,
    CO2_CSV_PATH,
    CO2_SOURCE_XLS_PATH,
]

missing_inputs = [
    path
    for path in REQUIRED_INPUTS
    if not path.is_file()
]

if missing_inputs:
    missing_text = "\n".join(
        str(path)
        for path in missing_inputs
    )
    raise FileNotFoundError(
        "以下必要输入文件不存在：\n"
        + missing_text
    )

# =========================
# 5. 可选检查来源XLS工作表
# =========================

xls_inspection = {
    "path": str(CO2_SOURCE_XLS_PATH),
    "exists": True,
    "size_bytes": int(
        CO2_SOURCE_XLS_PATH.stat().st_size
    ),
    "sha256": sha256_file(
        CO2_SOURCE_XLS_PATH
    ),
    "inspection_status": "not_attempted",
    "sheet_names": [],
    "error": None,
}

try:
    import xlrd

    xls_book = xlrd.open_workbook(
        str(CO2_SOURCE_XLS_PATH),
        on_demand=True,
    )

    xls_inspection["inspection_status"] = "passed"
    xls_inspection["sheet_names"] = (
        xls_book.sheet_names()
    )

    xls_book.release_resources()

except Exception as error:
    # 本阶段的精确匹配仅使用CSV，因此XLS检查失败不终止运行。
    xls_inspection["inspection_status"] = (
        "skipped_or_failed"
    )
    xls_inspection["error"] = (
        f"{type(error).__name__}: {error}"
    )

# =========================
# 6. 保存环境及输入审计
# =========================

environment_audit = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "run_code_version": RUN_CODE_VERSION,
    "python": sys.version,
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "rdkit_version": rdBase.rdkitVersion,
    "xlrd_version": get_package_version("xlrd"),
    "reference_temperature_K": REFERENCE_T_K,
    "reference_temperature_tolerance_K": (
        REFERENCE_T_TOL_K
    ),
    "input_files": [
        {
            "name": path.name,
            "path": str(path),
            "size_bytes": int(
                path.stat().st_size
            ),
            "sha256": sha256_file(path),
        }
        for path in REQUIRED_INPUTS
    ],
    "source_xls_inspection": xls_inspection,
    "output_root": str(OUT_DIR),
}

save_json_atomic(
    environment_audit,
    OUTPUT_PATHS["environment_input_audit"],
)

print("=" * 100)
print("Cell 0 环境、路径与输入文件审计完成 ✓")
print("=" * 100)
print("RDKit version             =", rdBase.rdkitVersion)
print("pandas version            =", pd.__version__)
print("xlrd inspection status    =", xls_inspection["inspection_status"])
print("output root               =", OUT_DIR)

if xls_inspection["sheet_names"]:
    print(
        "source XLS sheet count   =",
        len(xls_inspection["sheet_names"]),
    )
    print(
        "source XLS first sheets  =",
        xls_inspection["sheet_names"][:10],
    )

Cell 0 环境、路径与输入文件审计完成 ✓
RDKit version             = 2025.03.5
pandas version            = 2.3.1
xlrd inspection status    = passed
output root               = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck
source XLS sheet count   = 22
source XLS first sheets  = ['360QexF', 'Title', 'Table of Contents', 'Table S1', 'Table S2', 'Table S3', 'Table S4', 'Table S5', 'Table S6', 'Table S7']


In [2]:
# Cell 1 | 读取候选表、数据库及审计信息

candidate_df = pd.read_csv(
    CANDIDATE_PATH,
    low_memory=False,
)

reasonable_library_df = pd.read_csv(
    REASONABLE_LIBRARY_PATH,
    low_memory=False,
)

source_cation_df = pd.read_csv(
    SOURCE_CATION_PATH,
    low_memory=False,
)

source_anion_df = pd.read_csv(
    SOURCE_ANION_PATH,
    low_memory=False,
)

co2_df = pd.read_csv(
    CO2_CSV_PATH,
    low_memory=False,
)

generation_audit = read_json(
    GENERATION_AUDIT_PATH
)

reasonability_audit = read_json(
    REASONABILITY_AUDIT_PATH
)

prediction_manifest = read_json(
    PREDICTION_MANIFEST_PATH
)

run_manifest = read_json(
    RUN_MANIFEST_PATH
)

# 去除字段名前后可能存在的空格
for dataframe in [
    candidate_df,
    reasonable_library_df,
    source_cation_df,
    source_anion_df,
    co2_df,
]:
    dataframe.columns = (
        dataframe.columns
        .astype(str)
        .str.strip()
    )

candidate_required_columns = {
    "virtual_IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_pred_K",
    "P_pred_bar",
    "pred_lg_eta",
    "pred_eta_mPa_s",
    "rank_viscosity_nonhalide",
}

co2_required_columns = {
    "cation_smiles",
    "anion_smiles",
    "gas_smiles",
    "temperature",
    "pressure",
    "solubility",
}

missing_candidate_columns = (
    candidate_required_columns
    - set(candidate_df.columns)
)

missing_co2_columns = (
    co2_required_columns
    - set(co2_df.columns)
)

if missing_candidate_columns:
    raise KeyError(
        "候选表缺少字段："
        + str(
            sorted(missing_candidate_columns)
        )
    )

if missing_co2_columns:
    raise KeyError(
        "CO2数据库缺少字段："
        + str(
            sorted(missing_co2_columns)
        )
    )

# 增加稳定的原始记录编号
candidate_df.insert(
    0,
    "candidate_record_id",
    np.arange(
        1,
        len(candidate_df) + 1,
        dtype=int,
    ),
)

co2_df.insert(
    0,
    "co2_record_id",
    np.arange(
        1,
        len(co2_df) + 1,
        dtype=int,
    ),
)

# 数值字段统一转为数值型
candidate_numeric_columns = [
    "T_pred_K",
    "P_pred_bar",
    "pred_lg_eta",
    "pred_eta_mPa_s",
    "rank_viscosity_nonhalide",
]

for column in candidate_numeric_columns:
    candidate_df[column] = pd.to_numeric(
        candidate_df[column],
        errors="coerce",
    )

co2_numeric_columns = [
    "temperature",
    "pressure",
    "solubility",
]

for column in co2_numeric_columns:
    co2_df[column] = pd.to_numeric(
        co2_df[column],
        errors="coerce",
    )

# =========================
# 严格核对候选表
# =========================

expected_nonhalide_count = int(
    prediction_manifest[
        "nonhalide_candidates"
    ]
)

expected_reasonable_count = int(
    reasonability_audit[
        "after_filter"
    ]
)

expected_cation_count = int(
    generation_audit[
        "source_unique_cations"
    ]
)

expected_anion_count = int(
    generation_audit[
        "source_unique_anions"
    ]
)

if len(candidate_df) != expected_nonhalide_count:
    raise RuntimeError(
        "非简单卤素候选数量与预测manifest不一致："
        f"{len(candidate_df)} != "
        f"{expected_nonhalide_count}"
    )

if len(reasonable_library_df) != expected_reasonable_count:
    raise RuntimeError(
        "合理性筛选后候选数量与审计文件不一致。"
    )

if len(source_cation_df) != expected_cation_count:
    raise RuntimeError(
        "来源阳离子数量与生成审计不一致。"
    )

if len(source_anion_df) != expected_anion_count:
    raise RuntimeError(
        "来源阴离子数量与生成审计不一致。"
    )

if candidate_df["virtual_IL_key"].duplicated().any():
    raise RuntimeError(
        "非简单卤素候选表存在重复virtual_IL_key。"
    )

if (
    candidate_df[
        "rank_viscosity_nonhalide"
    ]
    .isna()
    .any()
):
    raise RuntimeError(
        "候选表存在缺失的非简单卤素排名。"
    )

expected_ranks = np.arange(
    1,
    len(candidate_df) + 1,
    dtype=int,
)

actual_ranks = (
    candidate_df[
        "rank_viscosity_nonhalide"
    ]
    .astype(int)
    .to_numpy()
)

if not np.array_equal(
    actual_ranks,
    expected_ranks,
):
    raise RuntimeError(
        "非简单卤素候选排名不是连续的1—17082。"
    )

# =========================
# 输入数据审计汇总
# =========================

input_audit_df = pd.DataFrame([
    {
        "dataset": "nonhalide_candidate_ranking",
        "rows": len(candidate_df),
        "columns": len(candidate_df.columns),
        "unique_cations": (
            candidate_df["cation_smiles"].nunique()
        ),
        "unique_anions": (
            candidate_df["anion_smiles"].nunique()
        ),
        "unique_pairs": (
            candidate_df["virtual_IL_key"].nunique()
        ),
        "missing_cation_smiles": (
            candidate_df["cation_smiles"].isna().sum()
        ),
        "missing_anion_smiles": (
            candidate_df["anion_smiles"].isna().sum()
        ),
    },
    {
        "dataset": "reasonability_passed_library",
        "rows": len(reasonable_library_df),
        "columns": len(reasonable_library_df.columns),
        "unique_cations": (
            reasonable_library_df[
                "cation_smiles"
            ].nunique()
        ),
        "unique_anions": (
            reasonable_library_df[
                "anion_smiles"
            ].nunique()
        ),
        "unique_pairs": (
            reasonable_library_df[
                "virtual_IL_key"
            ].nunique()
        ),
        "missing_cation_smiles": (
            reasonable_library_df[
                "cation_smiles"
            ].isna().sum()
        ),
        "missing_anion_smiles": (
            reasonable_library_df[
                "anion_smiles"
            ].isna().sum()
        ),
    },
    {
        "dataset": "CO2_solubility_database",
        "rows": len(co2_df),
        "columns": len(co2_df.columns),
        "unique_cations": (
            co2_df["cation_smiles"].nunique(
                dropna=True
            )
        ),
        "unique_anions": (
            co2_df["anion_smiles"].nunique(
                dropna=True
            )
        ),
        "unique_pairs": np.nan,
        "missing_cation_smiles": (
            co2_df["cation_smiles"].isna().sum()
        ),
        "missing_anion_smiles": (
            co2_df["anion_smiles"].isna().sum()
        ),
    },
])

save_csv_atomic(
    input_audit_df,
    OUTPUT_PATHS["input_dataset_audit"],
)

print("=" * 100)
print("Cell 1 输入数据读取与基础审计完成 ✓")
print("=" * 100)
print("nonhalide candidates       =", len(candidate_df))
print("reasonability passed       =", len(reasonable_library_df))
print("source unique cations      =", len(source_cation_df))
print("source unique anions       =", len(source_anion_df))
print("CO2 database rows          =", len(co2_df))

display(input_audit_df)

Cell 1 输入数据读取与基础审计完成 ✓
nonhalide candidates       = 17082
reasonability passed       = 17734
source unique cations      = 583
source unique anions       = 202
CO2 database rows          = 19342


,dataset,rows,columns,unique_cations,unique_anions,unique_pairs,missing_cation_smiles,missing_anion_smiles
0,nonhalide_candidate_ranking,17082,27,231,77,17082.0,0,0
1,reasonability_passed_library,17734,21,231,80,17734.0,0,0
2,CO2_solubility_database,19342,7,124,118,NaN,0,0


In [3]:
# Cell 2 | 统一规范化SMILES并筛选可匹配记录

def canonicalize_smiles(
    value,
):
    if pd.isna(value):
        return None, "missing"

    smiles = str(value).strip()

    if not smiles:
        return None, "missing"

    try:
        molecule = Chem.MolFromSmiles(
            smiles,
            sanitize=True,
        )

        if molecule is None:
            return None, "parse_failed"

        canonical_smiles = Chem.MolToSmiles(
            molecule,
            isomericSmiles=True,
            canonical=True,
        )

        return canonical_smiles, "ok"

    except Exception as error:
        return (
            None,
            f"exception_{type(error).__name__}",
        )


def canonicalize_series(
    series: pd.Series,
    canonical_column: str,
    status_column: str,
) -> pd.DataFrame:
    results = [
        canonicalize_smiles(value)
        for value in series
    ]

    return pd.DataFrame(
        results,
        columns=[
            canonical_column,
            status_column,
        ],
        index=series.index,
    )


# =========================
# 1. 候选SMILES重新规范化审计
# =========================

candidate_cation_result = canonicalize_series(
    candidate_df["cation_smiles"],
    "candidate_cation_smiles_canonical",
    "candidate_cation_smiles_status",
)

candidate_anion_result = canonicalize_series(
    candidate_df["anion_smiles"],
    "candidate_anion_smiles_canonical",
    "candidate_anion_smiles_status",
)

candidate_df = pd.concat(
    [
        candidate_df,
        candidate_cation_result,
        candidate_anion_result,
    ],
    axis=1,
)

candidate_df[
    "candidate_pair_key_canonical"
] = (
    candidate_df[
        "candidate_cation_smiles_canonical"
    ].astype(str)
    + "||"
    + candidate_df[
        "candidate_anion_smiles_canonical"
    ].astype(str)
)

candidate_df[
    "candidate_key_matches_canonical"
] = (
    candidate_df[
        "virtual_IL_key"
    ]
    .astype(str)
    .str.strip()
    == candidate_df[
        "candidate_pair_key_canonical"
    ]
)

candidate_invalid_mask = (
    candidate_df[
        "candidate_cation_smiles_status"
    ].ne("ok")
    | candidate_df[
        "candidate_anion_smiles_status"
    ].ne("ok")
)

if candidate_invalid_mask.any():
    invalid_candidate_preview = (
        candidate_df.loc[
            candidate_invalid_mask,
            [
                "candidate_record_id",
                "virtual_IL_key",
                "cation_smiles",
                "anion_smiles",
                "candidate_cation_smiles_status",
                "candidate_anion_smiles_status",
            ],
        ]
    )

    display(invalid_candidate_preview.head(20))

    raise RuntimeError(
        "正式非简单卤素候选中存在无法规范化的SMILES。"
    )

if (
    candidate_df[
        "candidate_pair_key_canonical"
    ]
    .duplicated()
    .any()
):
    raise RuntimeError(
        "重新规范化后出现重复候选完整离子对。"
    )

# =========================
# 2. CO2数据库SMILES规范化
# =========================

co2_cation_result = canonicalize_series(
    co2_df["cation_smiles"],
    "co2_cation_smiles_canonical",
    "co2_cation_smiles_status",
)

co2_anion_result = canonicalize_series(
    co2_df["anion_smiles"],
    "co2_anion_smiles_canonical",
    "co2_anion_smiles_status",
)

co2_gas_result = canonicalize_series(
    co2_df["gas_smiles"],
    "co2_gas_smiles_canonical",
    "co2_gas_smiles_status",
)

co2_df = pd.concat(
    [
        co2_df,
        co2_cation_result,
        co2_anion_result,
        co2_gas_result,
    ],
    axis=1,
)

co2_canonical_smiles, co2_reference_status = (
    canonicalize_smiles(
        CO2_REFERENCE_SMILES
    )
)

if co2_reference_status != "ok":
    raise RuntimeError(
        "标准CO2 SMILES无法完成规范化。"
    )

co2_df["gas_is_CO2"] = (
    co2_df[
        "co2_gas_smiles_canonical"
    ].eq(co2_canonical_smiles)
)

co2_df["numeric_fields_valid"] = (
    co2_df["temperature"].notna()
    & co2_df["pressure"].notna()
    & co2_df["solubility"].notna()
    & co2_df["temperature"].gt(0)
    & co2_df["pressure"].ge(0)
    & co2_df["solubility"].ge(0)
)


def build_exclusion_reason(row) -> str:
    reasons = []

    if row["co2_cation_smiles_status"] != "ok":
        reasons.append(
            "invalid_cation_smiles"
        )

    if row["co2_anion_smiles_status"] != "ok":
        reasons.append(
            "invalid_anion_smiles"
        )

    if row["co2_gas_smiles_status"] != "ok":
        reasons.append(
            "invalid_gas_smiles"
        )

    if (
        row["co2_gas_smiles_status"] == "ok"
        and not row["gas_is_CO2"]
    ):
        reasons.append(
            "gas_is_not_CO2"
        )

    if not row["numeric_fields_valid"]:
        reasons.append(
            "invalid_or_missing_numeric_fields"
        )

    return ";".join(reasons)


co2_df["exact_matching_exclusion_reason"] = (
    co2_df.apply(
        build_exclusion_reason,
        axis=1,
    )
)

co2_df["valid_for_exact_matching"] = (
    co2_df[
        "exact_matching_exclusion_reason"
    ].eq("")
)

invalid_smiles_mask = (
    co2_df[
        "co2_cation_smiles_status"
    ].ne("ok")
    | co2_df[
        "co2_anion_smiles_status"
    ].ne("ok")
    | co2_df[
        "co2_gas_smiles_status"
    ].ne("ok")
)

co2_invalid_smiles_df = (
    co2_df.loc[
        invalid_smiles_mask
    ]
    .copy()
)

co2_excluded_df = (
    co2_df.loc[
        ~co2_df[
            "valid_for_exact_matching"
        ]
    ]
    .copy()
)

canonicalization_audit_df = pd.DataFrame([
    {
        "dataset": "candidate",
        "field": "cation_smiles",
        "total_rows": len(candidate_df),
        "canonicalization_ok": int(
            candidate_df[
                "candidate_cation_smiles_status"
            ].eq("ok").sum()
        ),
        "canonicalization_failed": int(
            candidate_df[
                "candidate_cation_smiles_status"
            ].ne("ok").sum()
        ),
    },
    {
        "dataset": "candidate",
        "field": "anion_smiles",
        "total_rows": len(candidate_df),
        "canonicalization_ok": int(
            candidate_df[
                "candidate_anion_smiles_status"
            ].eq("ok").sum()
        ),
        "canonicalization_failed": int(
            candidate_df[
                "candidate_anion_smiles_status"
            ].ne("ok").sum()
        ),
    },
    {
        "dataset": "CO2_database",
        "field": "cation_smiles",
        "total_rows": len(co2_df),
        "canonicalization_ok": int(
            co2_df[
                "co2_cation_smiles_status"
            ].eq("ok").sum()
        ),
        "canonicalization_failed": int(
            co2_df[
                "co2_cation_smiles_status"
            ].ne("ok").sum()
        ),
    },
    {
        "dataset": "CO2_database",
        "field": "anion_smiles",
        "total_rows": len(co2_df),
        "canonicalization_ok": int(
            co2_df[
                "co2_anion_smiles_status"
            ].eq("ok").sum()
        ),
        "canonicalization_failed": int(
            co2_df[
                "co2_anion_smiles_status"
            ].ne("ok").sum()
        ),
    },
    {
        "dataset": "CO2_database",
        "field": "gas_smiles",
        "total_rows": len(co2_df),
        "canonicalization_ok": int(
            co2_df[
                "co2_gas_smiles_status"
            ].eq("ok").sum()
        ),
        "canonicalization_failed": int(
            co2_df[
                "co2_gas_smiles_status"
            ].ne("ok").sum()
        ),
    },
    {
        "dataset": "CO2_database",
        "field": "complete_record",
        "total_rows": len(co2_df),
        "canonicalization_ok": int(
            co2_df[
                "valid_for_exact_matching"
            ].sum()
        ),
        "canonicalization_failed": int(
            (
                ~co2_df[
                    "valid_for_exact_matching"
                ]
            ).sum()
        ),
    },
])

save_csv_atomic(
    canonicalization_audit_df,
    OUTPUT_PATHS["canonicalization_audit"],
)

save_csv_atomic(
    co2_invalid_smiles_df,
    OUTPUT_PATHS["invalid_smiles_records"],
)

save_csv_atomic(
    co2_excluded_df,
    OUTPUT_PATHS["excluded_records"],
)

print("=" * 100)
print("Cell 2 SMILES规范化与可匹配记录筛选完成 ✓")
print("=" * 100)
print(
    "candidate canonical key unchanged =",
    int(
        candidate_df[
            "candidate_key_matches_canonical"
        ].sum()
    ),
    "/",
    len(candidate_df),
)
print(
    "CO2 database valid records        =",
    int(
        co2_df[
            "valid_for_exact_matching"
        ].sum()
    ),
)
print(
    "CO2 database excluded records     =",
    int(
        (
            ~co2_df[
                "valid_for_exact_matching"
            ]
        ).sum()
    ),
)

display(canonicalization_audit_df)

[21:13:00] Explicit valence for atom # 3 N, 5, is greater than permitted
[21:13:00] Explicit valence for atom # 3 N, 5, is greater than permitted
[21:13:00] Explicit valence for atom # 3 N, 5, is greater than permitted
[21:13:00] Explicit valence for atom # 3 N, 5, is greater than permitted
[21:13:00] Explicit valence for atom # 3 N, 5, is greater than permitted
[21:13:00] Explicit valence for atom # 3 N, 5, is greater than permitted
[21:13:00] Explicit valence for atom # 3 N, 5, is greater than permitted
[21:13:02] Explicit valence for atom # 0 B, 4, is greater than permitted
[21:13:02] Explicit valence for atom # 0 B, 4, is greater than permitted
[21:13:02] Explicit valence for atom # 0 B, 4, is greater than permitted
[21:13:02] Explicit valence for atom # 0 B, 4, is greater than permitted
[21:13:02] Explicit valence for atom # 0 B, 4, is greater than permitted
[21:13:02] Explicit valence for atom # 0 B, 4, is greater than permitted
[21:13:02] Explicit valence for atom # 0 B, 4, is g

Cell 2 SMILES规范化与可匹配记录筛选完成 ✓
candidate canonical key unchanged = 17082 / 17082
CO2 database valid records        = 16672
CO2 database excluded records     = 2670


,dataset,field,total_rows,canonicalization_ok,canonicalization_failed
0,candidate,cation_smiles,17082,17082,0
1,candidate,anion_smiles,17082,17082,0
2,CO2_database,cation_smiles,19342,19335,7
3,CO2_database,anion_smiles,19342,16701,2641
4,CO2_database,gas_smiles,19342,19342,0
5,CO2_database,complete_record,19342,16672,2670


In [4]:
# Cell 3 | 构建完整离子液体匹配键

co2_valid_df = (
    co2_df.loc[
        co2_df[
            "valid_for_exact_matching"
        ]
    ]
    .copy()
)

co2_valid_df[
    "co2_pair_key_canonical"
] = (
    co2_valid_df[
        "co2_cation_smiles_canonical"
    ]
    + "||"
    + co2_valid_df[
        "co2_anion_smiles_canonical"
    ]
)

# 标记完全重复的实验记录，但暂不删除
co2_valid_df[
    "exact_duplicate_experimental_record"
] = co2_valid_df.duplicated(
    subset=[
        "co2_pair_key_canonical",
        "temperature",
        "pressure",
        "solubility",
    ],
    keep=False,
)

candidate_key_output_columns = [
    "candidate_record_id",
    "rank_viscosity_nonhalide",
    "virtual_IL_key",
    "candidate_pair_key_canonical",
    "candidate_key_matches_canonical",
    "cation_smiles",
    "anion_smiles",
    "candidate_cation_smiles_canonical",
    "candidate_anion_smiles_canonical",
    "T_pred_K",
    "P_pred_bar",
    "pred_lg_eta",
    "pred_eta_mPa_s",
]

candidate_key_df = (
    candidate_df[
        candidate_key_output_columns
    ]
    .copy()
)

co2_canonical_output_columns = [
    "co2_record_id",
    "cation_smiles",
    "anion_smiles",
    "gas_smiles",
    "co2_cation_smiles_canonical",
    "co2_anion_smiles_canonical",
    "co2_gas_smiles_canonical",
    "co2_pair_key_canonical",
    "temperature",
    "pressure",
    "solubility",
    "exact_duplicate_experimental_record",
]

co2_canonicalized_output_df = (
    co2_valid_df[
        co2_canonical_output_columns
    ]
    .copy()
)

save_csv_atomic(
    candidate_key_df,
    OUTPUT_PATHS["candidate_canonical_keys"],
)

save_csv_atomic(
    co2_canonicalized_output_df,
    OUTPUT_PATHS["co2_canonicalized_records"],
)

candidate_key_set = set(
    candidate_df[
        "candidate_pair_key_canonical"
    ]
)

co2_pair_key_set = set(
    co2_valid_df[
        "co2_pair_key_canonical"
    ]
)

exact_pair_key_intersection = (
    candidate_key_set
    & co2_pair_key_set
)

print("=" * 100)
print("Cell 3 完整离子液体匹配键构建完成 ✓")
print("=" * 100)
print("candidate unique pairs       =", len(candidate_key_set))
print("CO2 database unique pairs    =", len(co2_pair_key_set))
print(
    "exact pair-key intersection =",
    len(exact_pair_key_intersection),
)
print(
    "duplicate CO2 records       =",
    int(
        co2_valid_df[
            "exact_duplicate_experimental_record"
        ].sum()
    ),
)

Cell 3 完整离子液体匹配键构建完成 ✓
candidate unique pairs       = 17082
CO2 database unique pairs    = 315
exact pair-key intersection = 14
duplicate CO2 records       = 0


In [5]:
# Cell 4 | 读取最低5%正式候选并完成结构修正前的完整离子对精确匹配

from pathlib import Path
from datetime import datetime
import hashlib
import json
import os
import re
import tempfile

import numpy as np
import pandas as pd

from rdkit import Chem, RDLogger


RDLogger.DisableLog("rdApp.*")


# ============================================================
# 1. 正式分析口径与路径
# ============================================================

RUN_CODE_VERSION = (
    "04_lowest5pct_exact_pair_match_before_correction_v1_20260806"
)

TARGET_T_K = 298.15
TARGET_P_BAR = 1.01325
TEMPERATURE_WINDOW_K = 0.50

EXPECTED_LOWEST5_COUNT = 854
EXPECTED_ALL_NONHALIDE_COUNT = 17082

DATA_ROOT = Path(
    r"D:\jupyter\IL_ML_Project"
    r"\small_paper_data"
    r"\viscosity_10point_holdout"
)

VISCOSITY_OUTPUT_DIR = (
    DATA_ROOT
    / "outputs"
    / "virtual_IL_lightgbm_viscosity"
)

CO2_OUTPUT_DIR = (
    DATA_ROOT
    / "outputs"
    / "low_viscosity_CO2_solubility_crosscheck"
)

TABLE_DIR = CO2_OUTPUT_DIR / "tables"
LOG_DIR = CO2_OUTPUT_DIR / "logs"

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LOWEST5_CANDIDATE_PATH = (
    VISCOSITY_OUTPUT_DIR
    / "tables"
    / "virtual_IL_lowest5pct_nonhalide_candidates.csv"
)

ALL_NONHALIDE_CANDIDATE_PATH = (
    VISCOSITY_OUTPUT_DIR
    / "tables"
    / "virtual_IL_low_viscosity_ranking_nonhalide.csv"
)


# ============================================================
# 2. 通用文件函数
# ============================================================

def sha256_file(path):
    path = Path(path)

    hasher = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            hasher.update(chunk)

    return hasher.hexdigest()


def save_csv_atomic(
    dataframe,
    output_path,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8-sig",
        newline="",
        suffix=".csv",
        delete=False,
        dir=output_path.parent,
    ) as temp_file:
        temp_path = Path(
            temp_file.name
        )

    try:
        dataframe.to_csv(
            temp_path,
            index=False,
            encoding="utf-8-sig",
        )

        os.replace(
            temp_path,
            output_path,
        )

    finally:
        if temp_path.exists():
            temp_path.unlink()


def json_default(value):
    if isinstance(
        value,
        (
            np.integer,
            np.floating,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        Path,
    ):
        return str(value)

    if pd.isna(value):
        return None

    raise TypeError(
        f"无法序列化类型：{type(value)}"
    )


def save_json_atomic(
    data,
    output_path,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        suffix=".json",
        delete=False,
        dir=output_path.parent,
    ) as temp_file:
        temp_path = Path(
            temp_file.name
        )

        json.dump(
            data,
            temp_file,
            ensure_ascii=False,
            indent=2,
            default=json_default,
        )

    try:
        os.replace(
            temp_path,
            output_path,
        )

    finally:
        if temp_path.exists():
            temp_path.unlink()


# ============================================================
# 3. 列名和SMILES处理函数
# ============================================================

def normalize_column_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


def find_column(
    dataframe,
    aliases,
    required=True,
):
    normalized_columns = {
        normalize_column_name(column): column
        for column in dataframe.columns
    }

    for alias in aliases:
        normalized_alias = (
            normalize_column_name(
                alias
            )
        )

        if (
            normalized_alias
            in normalized_columns
        ):
            return normalized_columns[
                normalized_alias
            ]

    if required:
        raise KeyError(
            "没有找到所需字段。候选字段为："
            + ", ".join(
                map(
                    str,
                    aliases,
                )
            )
            + "\n当前字段为："
            + ", ".join(
                map(
                    str,
                    dataframe.columns,
                )
            )
        )

    return None


def canonicalize_smiles(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    if not text:
        return None

    try:
        molecule = Chem.MolFromSmiles(
            text
        )

        if molecule is None:
            return None

        return Chem.MolToSmiles(
            molecule,
            canonical=True,
            isomericSmiles=True,
        )

    except Exception:
        return None


def to_boolean_series(series):
    if pd.api.types.is_bool_dtype(
        series
    ):
        return series.fillna(
            False
        )

    normalized = (
        series.astype(str)
        .str.strip()
        .str.lower()
    )

    return normalized.isin({
        "true",
        "1",
        "yes",
        "y",
    })


# ============================================================
# 4. 标准化候选表
# ============================================================

def prepare_candidate_table(
    input_path,
    expected_count=None,
    expected_max_rank=None,
):
    input_path = Path(
        input_path
    )

    if not input_path.exists():
        raise FileNotFoundError(
            f"候选表不存在：\n{input_path}"
        )

    source_df = pd.read_csv(
        input_path,
        encoding="utf-8-sig",
        low_memory=False,
    )

    cation_column = find_column(
        source_df,
        [
            "cation_smiles",
            "candidate_cation_smiles",
            "cation_smiles_original",
        ],
    )

    anion_column = find_column(
        source_df,
        [
            "anion_smiles",
            "candidate_anion_smiles",
            "anion_smiles_original",
        ],
    )

    rank_column = find_column(
        source_df,
        [
            "rank_viscosity_nonhalide",
            "rank_nonhalide",
            "rank",
        ],
    )

    lg_eta_column = find_column(
        source_df,
        [
            "pred_lg_eta",
            "predicted_lg_eta",
        ],
    )

    eta_column = find_column(
        source_df,
        [
            "pred_eta_mPa_s",
            "predicted_eta_mPa_s",
        ],
    )

    temperature_column = find_column(
        source_df,
        [
            "T_pred_K",
            "temperature_K",
            "T_K",
        ],
        required=False,
    )

    pressure_column = find_column(
        source_df,
        [
            "P_pred_bar",
            "pressure_bar",
            "P_bar",
        ],
        required=False,
    )

    key_column = find_column(
        source_df,
        [
            "virtual_IL_key",
            "IL_key",
            "candidate_pair_key",
        ],
        required=False,
    )

    candidate_df = pd.DataFrame({
        "candidate_record_id": np.arange(
            1,
            len(source_df) + 1,
            dtype=int,
        ),
        "rank_viscosity_nonhalide": (
            pd.to_numeric(
                source_df[
                    rank_column
                ],
                errors="raise",
            )
            .astype(int)
        ),
        "candidate_cation_smiles_original": (
            source_df[
                cation_column
            ].astype(str)
        ),
        "candidate_anion_smiles_original": (
            source_df[
                anion_column
            ].astype(str)
        ),
        "pred_lg_eta": pd.to_numeric(
            source_df[
                lg_eta_column
            ],
            errors="raise",
        ),
        "pred_eta_mPa_s": pd.to_numeric(
            source_df[
                eta_column
            ],
            errors="raise",
        ),
    })

    if temperature_column is None:
        candidate_df[
            "T_pred_K"
        ] = TARGET_T_K
    else:
        candidate_df[
            "T_pred_K"
        ] = pd.to_numeric(
            source_df[
                temperature_column
            ],
            errors="coerce",
        )

    if pressure_column is None:
        candidate_df[
            "P_pred_bar"
        ] = TARGET_P_BAR
    else:
        candidate_df[
            "P_pred_bar"
        ] = pd.to_numeric(
            source_df[
                pressure_column
            ],
            errors="coerce",
        )

    candidate_df[
        "candidate_cation_smiles"
    ] = candidate_df[
        "candidate_cation_smiles_original"
    ].map(
        canonicalize_smiles
    )

    candidate_df[
        "candidate_anion_smiles"
    ] = candidate_df[
        "candidate_anion_smiles_original"
    ].map(
        canonicalize_smiles
    )

    if candidate_df[
        [
            "candidate_cation_smiles",
            "candidate_anion_smiles",
        ]
    ].isna().any().any():
        invalid_rows = candidate_df.loc[
            candidate_df[
                [
                    "candidate_cation_smiles",
                    "candidate_anion_smiles",
                ]
            ].isna().any(
                axis=1
            )
        ]

        raise RuntimeError(
            "正式候选表中存在无法规范化的SMILES：\n"
            + invalid_rows.head(
                20
            ).to_string(
                index=False
            )
        )

    candidate_df[
        "candidate_pair_key_canonical"
    ] = (
        candidate_df[
            "candidate_cation_smiles"
        ]
        + "||"
        + candidate_df[
            "candidate_anion_smiles"
        ]
    )

    if key_column is None:
        candidate_df[
            "virtual_IL_key"
        ] = candidate_df[
            "candidate_pair_key_canonical"
        ]
    else:
        candidate_df[
            "virtual_IL_key"
        ] = source_df[
            key_column
        ].astype(str)

    candidate_df = (
        candidate_df.sort_values(
            [
                "rank_viscosity_nonhalide",
                "candidate_pair_key_canonical",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    if expected_count is not None:
        if (
            len(candidate_df)
            != expected_count
        ):
            raise RuntimeError(
                f"候选数量应为{expected_count}，"
                f"实际为{len(candidate_df)}。"
            )

    if expected_max_rank is not None:
        expected_ranks = np.arange(
            1,
            expected_max_rank + 1,
            dtype=int,
        )

        actual_ranks = candidate_df[
            "rank_viscosity_nonhalide"
        ].to_numpy(
            dtype=int
        )

        if not np.array_equal(
            actual_ranks,
            expected_ranks,
        ):
            raise RuntimeError(
                "候选排名不是预期的连续整数范围。"
            )

    if candidate_df[
        "candidate_pair_key_canonical"
    ].duplicated().any():
        raise RuntimeError(
            "候选表中存在重复完整离子对。"
        )

    return candidate_df


# ============================================================
# 5. 从已有上游Cell中识别原始CO2溶解度DataFrame
# ============================================================

def resolve_original_co2_dataframe():
    preferred_names = [
        "co2_df",
        "co2_raw_df",
        "co2_data_df",
        "co2_solubility_df",
        "df_co2",
        "co2_working_df",
        "co2_source_df",
    ]

    candidates = []

    for variable_name, value in globals().items():
        if not isinstance(
            value,
            pd.DataFrame,
        ):
            continue

        try:
            cation_column = find_column(
                value,
                [
                    "cation_smiles",
                    "cation smiles",
                    "cation_SMILES",
                    "SMILES_cation",
                ],
                required=False,
            )

            anion_column = find_column(
                value,
                [
                    "anion_smiles",
                    "anion smiles",
                    "anion_SMILES",
                    "SMILES_anion",
                ],
                required=False,
            )

            solubility_column = find_column(
                value,
                [
                    "co2_solubility",
                    "solubility",
                    "mole_fraction",
                    "x_CO2",
                    "CO2_mole_fraction",
                ],
                required=False,
            )

            temperature_column = find_column(
                value,
                [
                    "co2_temperature_K",
                    "temperature_K",
                    "Temperature",
                    "T_K",
                ],
                required=False,
            )

            pressure_column = find_column(
                value,
                [
                    "co2_pressure_bar",
                    "pressure_bar",
                    "Pressure",
                    "P_bar",
                ],
                required=False,
            )

        except Exception:
            continue

        if (
            cation_column is not None
            and anion_column is not None
            and solubility_column is not None
            and temperature_column is not None
            and pressure_column is not None
        ):
            preferred_score = (
                1
                if variable_name
                in preferred_names
                else 0
            )

            candidates.append((
                preferred_score,
                len(value),
                variable_name,
                value,
            ))

    if not candidates:
        raise RuntimeError(
            "未能从Cell 0—Cell 3的内存变量中识别"
            "CO2溶解度数据库DataFrame。\n"
            "请确认已按顺序运行Cell 0—Cell 3。"
        )

    candidates.sort(
        key=lambda item: (
            item[0],
            item[1],
        ),
        reverse=True,
    )

    _, _, variable_name, dataframe = (
        candidates[0]
    )

    return (
        variable_name,
        dataframe.copy(),
    )


# ============================================================
# 6. 标准化CO2实验记录
# ============================================================

def prepare_co2_records(
    source_df,
    use_corrected_columns=False,
):
    if use_corrected_columns:
        cation_aliases = [
            "corrected_cation_smiles_canonical",
            "cation_smiles_canonical",
            "cation_smiles_working",
            "cation_smiles",
        ]

        anion_aliases = [
            "corrected_anion_smiles_canonical",
            "anion_smiles_canonical",
            "anion_smiles_working",
            "anion_smiles",
        ]

    else:
        cation_aliases = [
            "cation_smiles_original",
            "cation_smiles",
            "cation smiles",
            "cation_SMILES",
            "SMILES_cation",
        ]

        anion_aliases = [
            "anion_smiles_original",
            "anion_smiles",
            "anion smiles",
            "anion_SMILES",
            "SMILES_anion",
        ]

    cation_column = find_column(
        source_df,
        cation_aliases,
    )

    anion_column = find_column(
        source_df,
        anion_aliases,
    )

    temperature_column = find_column(
        source_df,
        [
            "co2_temperature_K",
            "temperature_K",
            "Temperature",
            "T_K",
            "temperature",
        ],
    )

    pressure_column = find_column(
        source_df,
        [
            "co2_pressure_bar",
            "pressure_bar",
            "Pressure",
            "P_bar",
            "pressure",
        ],
    )

    solubility_column = find_column(
        source_df,
        [
            "co2_solubility",
            "solubility",
            "mole_fraction",
            "x_CO2",
            "CO2_mole_fraction",
        ],
    )

    record_id_column = find_column(
        source_df,
        [
            "co2_record_id",
            "record_id",
            "index",
        ],
        required=False,
    )

    correction_applied_column = (
        find_column(
            source_df,
            [
                "structure_correction_applied",
                "correction_applied",
            ],
            required=False,
        )
    )

    correction_rule_column = (
        find_column(
            source_df,
            [
                "applied_correction_rule_ids",
                "correction_rule_ids",
            ],
            required=False,
        )
    )

    records_df = pd.DataFrame({
        "co2_record_id": (
            source_df[
                record_id_column
            ]
            if record_id_column
            is not None
            else np.arange(
                1,
                len(source_df) + 1,
                dtype=int,
            )
        ),
        "cation_smiles_original": (
            source_df[
                cation_column
            ].astype(str)
        ),
        "anion_smiles_original": (
            source_df[
                anion_column
            ].astype(str)
        ),
        "co2_temperature_K": (
            pd.to_numeric(
                source_df[
                    temperature_column
                ],
                errors="coerce",
            )
        ),
        "co2_pressure_bar": (
            pd.to_numeric(
                source_df[
                    pressure_column
                ],
                errors="coerce",
            )
        ),
        "co2_solubility": (
            pd.to_numeric(
                source_df[
                    solubility_column
                ],
                errors="coerce",
            )
        ),
    })

    records_df[
        "corrected_cation_smiles_canonical"
    ] = records_df[
        "cation_smiles_original"
    ].map(
        canonicalize_smiles
    )

    records_df[
        "corrected_anion_smiles_canonical"
    ] = records_df[
        "anion_smiles_original"
    ].map(
        canonicalize_smiles
    )

    records_df[
        "corrected_pair_key_canonical"
    ] = np.where(
        records_df[
            "corrected_cation_smiles_canonical"
        ].notna()
        & records_df[
            "corrected_anion_smiles_canonical"
        ].notna(),
        (
            records_df[
                "corrected_cation_smiles_canonical"
            ]
            + "||"
            + records_df[
                "corrected_anion_smiles_canonical"
            ]
        ),
        None,
    )

    if correction_applied_column is None:
        records_df[
            "structure_correction_applied"
        ] = False
    else:
        records_df[
            "structure_correction_applied"
        ] = to_boolean_series(
            source_df[
                correction_applied_column
            ]
        )

    if correction_rule_column is None:
        records_df[
            "applied_correction_rule_ids"
        ] = ""
    else:
        records_df[
            "applied_correction_rule_ids"
        ] = (
            source_df[
                correction_rule_column
            ]
            .fillna("")
            .astype(str)
        )

    records_df[
        "absolute_temperature_difference_to_29815_K"
    ] = (
        records_df[
            "co2_temperature_K"
        ]
        - TARGET_T_K
    ).abs()

    records_df[
        "absolute_pressure_difference_to_101325_bar"
    ] = (
        records_df[
            "co2_pressure_bar"
        ]
        - TARGET_P_BAR
    ).abs()

    records_df[
        "within_29815K_window"
    ] = (
        records_df[
            "absolute_temperature_difference_to_29815_K"
        ]
        <= TEMPERATURE_WINDOW_K
    )

    return records_df


# ============================================================
# 7. 精确匹配和候选汇总函数
# ============================================================

def exact_pair_merge(
    candidate_df,
    co2_records_df,
):
    valid_records_df = (
        co2_records_df.loc[
            co2_records_df[
                "corrected_pair_key_canonical"
            ].notna()
        ]
        .copy()
    )

    matched_df = candidate_df.merge(
        valid_records_df,
        left_on=(
            "candidate_pair_key_canonical"
        ),
        right_on=(
            "corrected_pair_key_canonical"
        ),
        how="inner",
        validate="one_to_many",
    )

    return matched_df


def build_exact_match_summary(
    matched_df,
):
    summary_columns = [
        "candidate_pair_key_canonical",
        "rank_viscosity_nonhalide",
        "candidate_cation_smiles",
        "candidate_anion_smiles",
        "pred_lg_eta",
        "pred_eta_mPa_s",
        "n_CO2_records",
        "n_unique_temperatures",
        "temperature_min_K",
        "temperature_max_K",
        "pressure_min_bar",
        "pressure_max_bar",
        "solubility_min",
        "solubility_max",
        "n_corrected_source_records",
        "n_records_within_29815K_window",
    ]

    if matched_df.empty:
        return pd.DataFrame(
            columns=summary_columns
        )

    summary_df = (
        matched_df.groupby(
            [
                "candidate_pair_key_canonical",
                "rank_viscosity_nonhalide",
                "candidate_cation_smiles",
                "candidate_anion_smiles",
                "pred_lg_eta",
                "pred_eta_mPa_s",
            ],
            as_index=False,
            dropna=False,
        )
        .agg(
            n_CO2_records=(
                "co2_record_id",
                "size",
            ),
            n_unique_temperatures=(
                "co2_temperature_K",
                "nunique",
            ),
            temperature_min_K=(
                "co2_temperature_K",
                "min",
            ),
            temperature_max_K=(
                "co2_temperature_K",
                "max",
            ),
            pressure_min_bar=(
                "co2_pressure_bar",
                "min",
            ),
            pressure_max_bar=(
                "co2_pressure_bar",
                "max",
            ),
            solubility_min=(
                "co2_solubility",
                "min",
            ),
            solubility_max=(
                "co2_solubility",
                "max",
            ),
            n_corrected_source_records=(
                "structure_correction_applied",
                "sum",
            ),
            n_records_within_29815K_window=(
                "within_29815K_window",
                "sum",
            ),
        )
        .sort_values(
            [
                "rank_viscosity_nonhalide",
                "candidate_pair_key_canonical",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    return summary_df[
        summary_columns
    ]


# ============================================================
# 8. 读取854种正式候选
# ============================================================

lowest5_candidates_df = (
    prepare_candidate_table(
        LOWEST5_CANDIDATE_PATH,
        expected_count=(
            EXPECTED_LOWEST5_COUNT
        ),
        expected_max_rank=(
            EXPECTED_LOWEST5_COUNT
        ),
    )
)


# ============================================================
# 9. 读取原始CO2数据库并完成修正前精确匹配
# ============================================================

(
    original_co2_variable_name,
    original_co2_source_df,
) = resolve_original_co2_dataframe()

co2_records_before_df = (
    prepare_co2_records(
        original_co2_source_df,
        use_corrected_columns=False,
    )
)

lowest5_exact_before_df = (
    exact_pair_merge(
        lowest5_candidates_df,
        co2_records_before_df,
    )
)

lowest5_exact_summary_before_df = (
    build_exact_match_summary(
        lowest5_exact_before_df
    )
)


# ============================================================
# 10. 保存Cell 4结果
# ============================================================

LOWEST5_EXACT_RECORDS_BEFORE_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "exact_pair_matched_records_before_correction.csv"
    )
)

LOWEST5_EXACT_SUMMARY_BEFORE_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "exact_pair_match_summary_before_correction.csv"
    )
)

save_csv_atomic(
    lowest5_exact_before_df,
    LOWEST5_EXACT_RECORDS_BEFORE_PATH,
)

save_csv_atomic(
    lowest5_exact_summary_before_df,
    LOWEST5_EXACT_SUMMARY_BEFORE_PATH,
)


# ============================================================
# 11. 基本审计
# ============================================================

exact_matched_candidates_before = (
    lowest5_exact_before_df[
        "candidate_pair_key_canonical"
    ].nunique()
)

if (
    exact_matched_candidates_before
    != 0
):
    raise RuntimeError(
        "结构修正前的854种最低5%候选中"
        "出现了完整离子对精确匹配，"
        "与当前已确认结果不一致。"
    )

print(
    "=" * 96
)

print(
    "Cell 4 最低5%候选结构修正前"
    "完整离子对精确匹配完成 ✓"
)

print(
    "=" * 96
)

print(
    "正式候选数量                   =",
    len(
        lowest5_candidates_df
    ),
)

print(
    "CO2数据库内存变量              =",
    original_co2_variable_name,
)

print(
    "原始CO2数据库记录数            =",
    len(
        original_co2_source_df
    ),
)

print(
    "可建立完整离子对键的记录数     =",
    int(
        co2_records_before_df[
            "corrected_pair_key_canonical"
        ].notna().sum()
    ),
)

print(
    "精确匹配完整候选数             =",
    exact_matched_candidates_before,
)

print(
    "精确匹配实验记录数             =",
    len(
        lowest5_exact_before_df
    ),
)

print(
    "匹配记录输出                   =",
    LOWEST5_EXACT_RECORDS_BEFORE_PATH,
)

print(
    "候选汇总输出                   =",
    LOWEST5_EXACT_SUMMARY_BEFORE_PATH,
)

display(
    lowest5_exact_summary_before_df
)

Cell 4 最低5%候选结构修正前完整离子对精确匹配完成 ✓
正式候选数量                   = 854
CO2数据库内存变量              = co2_df
原始CO2数据库记录数            = 19342
可建立完整离子对键的记录数     = 16694
精确匹配完整候选数             = 0
精确匹配实验记录数             = 0
匹配记录输出                   = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\lowest5pct_candidate_CO2_exact_pair_matched_records_before_correction.csv
候选汇总输出                   = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\lowest5pct_candidate_CO2_exact_pair_match_summary_before_correction.csv


,candidate_pair_key_canonical,rank_viscosity_nonhalide,candidate_cation_smiles,candidate_anion_smiles,pred_lg_eta,pred_eta_mPa_s,n_CO2_records,n_unique_temperatures,temperature_min_K,temperature_max_K,pressure_min_bar,pressure_max_bar,solubility_min,solubility_max,n_corrected_source_records,n_records_within_29815K_window


In [6]:
# Cell 5 | 汇总最低5%候选结构修正前精确匹配及温度覆盖审计

CELL5_RUN_CODE_VERSION = (
    "05_lowest5pct_exact_pair_match_before_correction_audit_v1_20260806"
)


# ============================================================
# 1. 构建最低5%单一正式范围统计
# ============================================================

lowest5_exact_T_window_before_df = (
    lowest5_exact_before_df.loc[
        lowest5_exact_before_df[
            "within_29815K_window"
        ]
    ]
    .copy()
    .sort_values(
        [
            "rank_viscosity_nonhalide",
            "absolute_temperature_difference_to_29815_K",
            "absolute_pressure_difference_to_101325_bar",
            "co2_record_id",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

lowest5_exact_scope_before_df = pd.DataFrame([
    {
        "scope": (
            "Lowest5pct_nonhalide"
        ),
        "rank_limit": (
            EXPECTED_LOWEST5_COUNT
        ),
        "total_candidate_count": int(
            len(
                lowest5_candidates_df
            )
        ),
        "exact_matched_candidate_count": int(
            lowest5_exact_before_df[
                "candidate_pair_key_canonical"
            ].nunique()
        ),
        "exact_matched_candidate_fraction": (
            float(
                lowest5_exact_before_df[
                    "candidate_pair_key_canonical"
                ].nunique()
            )
            / float(
                len(
                    lowest5_candidates_df
                )
            )
        ),
        "matched_experimental_record_count": int(
            len(
                lowest5_exact_before_df
            )
        ),
        "matched_records_within_29815K_window": int(
            len(
                lowest5_exact_T_window_before_df
            )
        ),
    }
])


# ============================================================
# 2. 保存正式审计文件
# ============================================================

LOWEST5_EXACT_SCOPE_BEFORE_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "exact_match_counts_before_correction.csv"
    )
)

LOWEST5_EXACT_T_WINDOW_BEFORE_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "exact_match_T29815_window_before_correction.csv"
    )
)

LOWEST5_EXACT_BEFORE_MANIFEST_PATH = (
    LOG_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "exact_match_before_correction_manifest.json"
    )
)

save_csv_atomic(
    lowest5_exact_scope_before_df,
    LOWEST5_EXACT_SCOPE_BEFORE_PATH,
)

save_csv_atomic(
    lowest5_exact_T_window_before_df,
    LOWEST5_EXACT_T_WINDOW_BEFORE_PATH,
)

lowest5_exact_before_manifest = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "run_code_version": (
        CELL5_RUN_CODE_VERSION
    ),
    "status": (
        "lowest5pct_exact_pair_matching_"
        "before_structure_correction_completed"
    ),
    "analysis_scope": {
        "candidate_scope": (
            "lowest_5pct_non_simple_halide_candidates"
        ),
        "candidate_count": int(
            len(
                lowest5_candidates_df
            )
        ),
        "rank_range": [
            int(
                lowest5_candidates_df[
                    "rank_viscosity_nonhalide"
                ].min()
            ),
            int(
                lowest5_candidates_df[
                    "rank_viscosity_nonhalide"
                ].max()
            ),
        ],
        "comparison_basis": (
            "exact_canonical_complete_ion_pair_identity"
        ),
        "temperature_window_center_K": (
            TARGET_T_K
        ),
        "temperature_window_half_width_K": (
            TEMPERATURE_WINDOW_K
        ),
        "molecular_fingerprint_used": False,
        "similarity_threshold_applied": False,
        "Henry_constant_calculated": False,
        "different_temperatures_combined": False,
        "different_pressures_combined": False,
    },
    "counts": {
        "candidate_count": int(
            len(
                lowest5_candidates_df
            )
        ),
        "original_CO2_database_rows": int(
            len(
                original_co2_source_df
            )
        ),
        "valid_complete_pair_records": int(
            co2_records_before_df[
                "corrected_pair_key_canonical"
            ].notna().sum()
        ),
        "exact_matched_candidates": int(
            lowest5_exact_before_df[
                "candidate_pair_key_canonical"
            ].nunique()
        ),
        "matched_experimental_records": int(
            len(
                lowest5_exact_before_df
            )
        ),
        "records_within_29815K_window": int(
            len(
                lowest5_exact_T_window_before_df
            )
        ),
    },
    "verification": {
        "candidate_count_is_854": bool(
            len(
                lowest5_candidates_df
            )
            == EXPECTED_LOWEST5_COUNT
        ),
        "candidate_ranks_are_1_to_854": bool(
            np.array_equal(
                lowest5_candidates_df[
                    "rank_viscosity_nonhalide"
                ].to_numpy(
                    dtype=int
                ),
                np.arange(
                    1,
                    EXPECTED_LOWEST5_COUNT + 1,
                    dtype=int,
                ),
            )
        ),
        "exact_match_count_is_zero": bool(
            lowest5_exact_before_df[
                "candidate_pair_key_canonical"
            ].nunique()
            == 0
        ),
        "original_CO2_database_modified": False,
        "different_temperatures_combined": False,
        "different_pressures_combined": False,
    },
    "inputs": {
        "lowest5pct_candidate_table": {
            "path": str(
                LOWEST5_CANDIDATE_PATH
            ),
            "sha256": sha256_file(
                LOWEST5_CANDIDATE_PATH
            ),
        },
        "original_CO2_dataframe_variable": (
            original_co2_variable_name
        ),
    },
    "outputs": {
        "matched_records": str(
            LOWEST5_EXACT_RECORDS_BEFORE_PATH
        ),
        "match_summary": str(
            LOWEST5_EXACT_SUMMARY_BEFORE_PATH
        ),
        "scope_summary": str(
            LOWEST5_EXACT_SCOPE_BEFORE_PATH
        ),
        "T29815_window_records": str(
            LOWEST5_EXACT_T_WINDOW_BEFORE_PATH
        ),
    },
}

save_json_atomic(
    lowest5_exact_before_manifest,
    LOWEST5_EXACT_BEFORE_MANIFEST_PATH,
)


# ============================================================
# 3. 输出结果
# ============================================================

print(
    "=" * 96
)

print(
    "Cell 5 最低5%候选结构修正前"
    "精确匹配审计完成 ✓"
)

print(
    "=" * 96
)

print(
    "candidate scope                 =",
    "Lowest5pct_nonhalide",
)

print(
    "candidate count                 =",
    len(
        lowest5_candidates_df
    ),
)

print(
    "exact matched candidates        =",
    lowest5_exact_before_df[
        "candidate_pair_key_canonical"
    ].nunique(),
)

print(
    "matched experimental records    =",
    len(
        lowest5_exact_before_df
    ),
)

print(
    "records within 298.15±0.50 K    =",
    len(
        lowest5_exact_T_window_before_df
    ),
)

print(
    "scope summary                   =",
    LOWEST5_EXACT_SCOPE_BEFORE_PATH,
)

print(
    "temperature-window records      =",
    LOWEST5_EXACT_T_WINDOW_BEFORE_PATH,
)

print(
    "manifest                        =",
    LOWEST5_EXACT_BEFORE_MANIFEST_PATH,
)

display(
    lowest5_exact_scope_before_df
)

Cell 5 最低5%候选结构修正前精确匹配审计完成 ✓
candidate scope                 = Lowest5pct_nonhalide
candidate count                 = 854
exact matched candidates        = 0
matched experimental records    = 0
records within 298.15±0.50 K    = 0
scope summary                   = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\lowest5pct_candidate_CO2_exact_match_counts_before_correction.csv
temperature-window records      = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\lowest5pct_candidate_CO2_exact_match_T29815_window_before_correction.csv
manifest                        = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\logs\lowest5pct_candidate_CO2_exact_match_before_correction_manifest.json


,scope,rank_limit,total_candidate_count,exact_matched_candidate_count,exact_matched_candidate_fraction,matched_experimental_record_count,matched_records_within_29815K_window
0,Lowest5pct_nonhalide,854,854,0,0.0,0,0


In [7]:
# Cell 6 | 汇总CO2数据库中无法规范化的唯一SMILES

from datetime import datetime

INVALID_UNIQUE_SUMMARY_PATH = (
    TABLE_DIR
    / "co2_database_invalid_SMILES_unique_summary.csv"
)

INVALID_ANION_SUMMARY_PATH = (
    TABLE_DIR
    / "co2_database_invalid_anion_SMILES_frequency.csv"
)

INVALID_CATION_SUMMARY_PATH = (
    TABLE_DIR
    / "co2_database_invalid_cation_SMILES_frequency.csv"
)

MANUAL_CORRECTION_TEMPLATE_PATH = (
    TABLE_DIR
    / "co2_database_invalid_SMILES_manual_correction_template.csv"
)

INVALID_SUMMARY_MANIFEST_PATH = (
    LOG_DIR
    / "co2_database_invalid_SMILES_summary_manifest.json"
)


def build_invalid_smiles_summary(
    dataframe: pd.DataFrame,
    ion_role: str,
    raw_smiles_column: str,
    status_column: str,
    counterion_column: str,
) -> pd.DataFrame:
    """按唯一原始SMILES汇总无法规范化的数据库记录。"""

    invalid_df = dataframe.loc[
        dataframe[status_column].ne("ok")
    ].copy()

    if invalid_df.empty:
        return pd.DataFrame(
            columns=[
                "ion_role",
                "raw_smiles",
                "parse_status",
                "record_count",
                "first_record_id",
                "last_record_id",
                "unique_counterion_count",
                "temperature_min_K",
                "temperature_max_K",
                "pressure_min_bar",
                "pressure_max_bar",
                "solubility_min",
                "solubility_max",
            ]
        )

    invalid_df["raw_smiles_for_summary"] = (
        invalid_df[raw_smiles_column]
        .astype("string")
        .fillna("<MISSING>")
        .str.strip()
    )

    summary_df = (
        invalid_df.groupby(
            [
                "raw_smiles_for_summary",
                status_column,
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            record_count=(
                "co2_record_id",
                "size",
            ),
            first_record_id=(
                "co2_record_id",
                "min",
            ),
            last_record_id=(
                "co2_record_id",
                "max",
            ),
            unique_counterion_count=(
                counterion_column,
                "nunique",
            ),
            temperature_min_K=(
                "temperature",
                "min",
            ),
            temperature_max_K=(
                "temperature",
                "max",
            ),
            pressure_min_bar=(
                "pressure",
                "min",
            ),
            pressure_max_bar=(
                "pressure",
                "max",
            ),
            solubility_min=(
                "solubility",
                "min",
            ),
            solubility_max=(
                "solubility",
                "max",
            ),
        )
        .rename(
            columns={
                "raw_smiles_for_summary": "raw_smiles",
                status_column: "parse_status",
            }
        )
    )

    summary_df.insert(
        0,
        "ion_role",
        ion_role,
    )

    summary_df = (
        summary_df.sort_values(
            by=[
                "record_count",
                "raw_smiles",
            ],
            ascending=[
                False,
                True,
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    return summary_df


invalid_cation_summary_df = (
    build_invalid_smiles_summary(
        dataframe=co2_df,
        ion_role="cation",
        raw_smiles_column="cation_smiles",
        status_column="co2_cation_smiles_status",
        counterion_column="anion_smiles",
    )
)

invalid_anion_summary_df = (
    build_invalid_smiles_summary(
        dataframe=co2_df,
        ion_role="anion",
        raw_smiles_column="anion_smiles",
        status_column="co2_anion_smiles_status",
        counterion_column="cation_smiles",
    )
)

invalid_unique_summary_df = pd.concat(
    [
        invalid_cation_summary_df,
        invalid_anion_summary_df,
    ],
    ignore_index=True,
)

invalid_unique_summary_df = (
    invalid_unique_summary_df.sort_values(
        by=[
            "ion_role",
            "record_count",
            "raw_smiles",
        ],
        ascending=[
            True,
            False,
            True,
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

# 建立人工核对模板，但本Cell不填写或执行任何修正。
manual_correction_template_df = (
    invalid_unique_summary_df[
        [
            "ion_role",
            "raw_smiles",
            "parse_status",
            "record_count",
            "unique_counterion_count",
        ]
    ]
    .copy()
)

manual_correction_template_df[
    "proposed_corrected_smiles"
] = ""

manual_correction_template_df[
    "chemical_identity"
] = ""

manual_correction_template_df[
    "correction_decision"
] = "pending_review"

manual_correction_template_df[
    "correction_basis"
] = ""

manual_correction_template_df[
    "notes"
] = ""

save_csv_atomic(
    invalid_unique_summary_df,
    INVALID_UNIQUE_SUMMARY_PATH,
)

save_csv_atomic(
    invalid_anion_summary_df,
    INVALID_ANION_SUMMARY_PATH,
)

save_csv_atomic(
    invalid_cation_summary_df,
    INVALID_CATION_SUMMARY_PATH,
)

save_csv_atomic(
    manual_correction_template_df,
    MANUAL_CORRECTION_TEMPLATE_PATH,
)

invalid_summary_manifest = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "run_code_version": (
        "02_invalid_CO2_SMILES_summary_v1_20260805"
    ),
    "status": (
        "invalid_CO2_SMILES_unique_summary_completed"
    ),
    "source_database_rows": int(
        len(co2_df)
    ),
    "invalid_record_counts": {
        "cation_records": int(
            co2_df[
                "co2_cation_smiles_status"
            ].ne("ok").sum()
        ),
        "anion_records": int(
            co2_df[
                "co2_anion_smiles_status"
            ].ne("ok").sum()
        ),
    },
    "invalid_unique_SMILES_counts": {
        "unique_cation_SMILES": int(
            len(invalid_cation_summary_df)
        ),
        "unique_anion_SMILES": int(
            len(invalid_anion_summary_df)
        ),
        "all_unique_failed_SMILES": int(
            len(invalid_unique_summary_df)
        ),
    },
    "automatic_correction_performed": False,
    "original_database_modified": False,
    "exact_matching_rerun": False,
    "outputs": {
        "all_unique_summary": {
            "path": str(
                INVALID_UNIQUE_SUMMARY_PATH
            ),
            "sha256": sha256_file(
                INVALID_UNIQUE_SUMMARY_PATH
            ),
        },
        "invalid_anion_summary": {
            "path": str(
                INVALID_ANION_SUMMARY_PATH
            ),
            "sha256": sha256_file(
                INVALID_ANION_SUMMARY_PATH
            ),
        },
        "invalid_cation_summary": {
            "path": str(
                INVALID_CATION_SUMMARY_PATH
            ),
            "sha256": sha256_file(
                INVALID_CATION_SUMMARY_PATH
            ),
        },
        "manual_correction_template": {
            "path": str(
                MANUAL_CORRECTION_TEMPLATE_PATH
            ),
            "sha256": sha256_file(
                MANUAL_CORRECTION_TEMPLATE_PATH
            ),
        },
    },
}

save_json_atomic(
    invalid_summary_manifest,
    INVALID_SUMMARY_MANIFEST_PATH,
)

print("=" * 100)
print("Cell 6 无法规范化的唯一SMILES汇总完成 ✓")
print("=" * 100)

print(
    "invalid cation records       =",
    invalid_summary_manifest[
        "invalid_record_counts"
    ][
        "cation_records"
    ],
)

print(
    "invalid anion records        =",
    invalid_summary_manifest[
        "invalid_record_counts"
    ][
        "anion_records"
    ],
)

print(
    "unique invalid cation SMILES =",
    len(invalid_cation_summary_df),
)

print(
    "unique invalid anion SMILES  =",
    len(invalid_anion_summary_df),
)

print("\n记录数最多的无效阴离子SMILES：")
display(
    invalid_anion_summary_df.head(30)
)

print("\n全部无效阳离子SMILES：")
display(
    invalid_cation_summary_df
)

print("\n输出文件：")
print(INVALID_UNIQUE_SUMMARY_PATH)
print(INVALID_ANION_SUMMARY_PATH)
print(INVALID_CATION_SUMMARY_PATH)
print(MANUAL_CORRECTION_TEMPLATE_PATH)
print(INVALID_SUMMARY_MANIFEST_PATH)

Cell 6 无法规范化的唯一SMILES汇总完成 ✓
invalid cation records       = 7
invalid anion records        = 2641
unique invalid cation SMILES = 1
unique invalid anion SMILES  = 9

记录数最多的无效阴离子SMILES：


,ion_role,raw_smiles,parse_status,record_count,first_record_id,last_record_id,unique_counterion_count,temperature_min_K,temperature_max_K,pressure_min_bar,pressure_max_bar,solubility_min,solubility_max
0,anion,B(F)(F)(F)F,parse_failed,953,231,7250,7,278.47,383.15,0.00,866.00,-0.0010,0.7523
1,anion,FP(F)(F)(F)(F)F,parse_failed,810,1197,7313,5,282.05,393.15,0.00,971.00,-0.0004,0.7550
2,anion,[CL-],parse_failed,314,716,16607,6,302.55,373.15,1.68,369.46,0.1190,0.8924
3,anion,P(F)(F)(F)(F)(F)F,parse_failed,204,5042,5745,2,298.15,373.15,1.33,946.00,0.0060,0.7270
4,anion,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,parse_failed,140,7754,7893,1,303.15,363.15,3.90,113.70,0.0971,0.7748
5,anion,[BR-],parse_failed,124,5406,19052,6,293.15,363.44,1.00,148.91,0.0741,0.6970
6,anion,B(C#N)(C#N)(C#N)C#N,parse_failed,62,5792,5853,1,283.56,364.04,2.71,123.36,0.1009,0.7016
7,anion,BRC=1C=CC2=C(N=C[N-]2)C1,parse_failed,26,18889,18984,2,295.15,333.15,0.01,1.02,0.1150,0.4737
8,anion,C1=CC(=O)NC1=[O-],parse_failed,8,5784,5791,1,333.27,333.27,0.02,3.00,0.0000,0.0258



全部无效阳离子SMILES：


,ion_role,raw_smiles,parse_status,record_count,first_record_id,last_record_id,unique_counterion_count,temperature_min_K,temperature_max_K,pressure_min_bar,pressure_max_bar,solubility_min,solubility_max
0,cation,CCC[NH3+]C(C)C,parse_failed,7,6935,6941,1,304.16,344.84,0.47,0.54,0.0122,0.0238



输出文件：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\co2_database_invalid_SMILES_unique_summary.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\co2_database_invalid_anion_SMILES_frequency.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\co2_database_invalid_cation_SMILES_frequency.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\co2_database_invalid_SMILES_manual_correction_template.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\logs\co2_database_invalid_SMILES_summary_manifest.json


In [8]:
# Cell 7 | 高置信度SMILES修正规则审计与来源工作簿追溯

from datetime import datetime
import json
import re

import pandas as pd
import xlrd
from rdkit import Chem


# ============================================================
# 1. 输出路径
# ============================================================

HIGH_CONFIDENCE_AUDIT_PATH = (
    TABLE_DIR
    / "co2_invalid_SMILES_high_confidence_correction_audit.csv"
)

UNRESOLVED_SMILES_PATH = (
    TABLE_DIR
    / "co2_invalid_SMILES_unresolved_for_source_review.csv"
)

SOURCE_XLS_SHEET_INVENTORY_PATH = (
    TABLE_DIR
    / "co2_source_workbook_sheet_inventory.csv"
)

SOURCE_XLS_EXACT_HITS_PATH = (
    TABLE_DIR
    / "co2_source_workbook_invalid_SMILES_exact_hits.csv"
)

SOURCE_XLS_HEADER_CANDIDATES_PATH = (
    TABLE_DIR
    / "co2_source_workbook_header_candidates.csv"
)

CELL7_MANIFEST_PATH = (
    LOG_DIR
    / "co2_invalid_SMILES_correction_and_source_trace_manifest.json"
)


# ============================================================
# 2. 高置信度修正规则
# ============================================================

HIGH_CONFIDENCE_CORRECTIONS = [
    {
        "ion_role": "anion",
        "raw_smiles": "B(F)(F)(F)F",
        "proposed_corrected_smiles": "[B-](F)(F)(F)F",
        "chemical_identity": "tetrafluoroborate",
        "correction_basis": (
            "四配位硼酸根原始写法缺少形式负电荷"
        ),
    },
    {
        "ion_role": "anion",
        "raw_smiles": "FP(F)(F)(F)(F)F",
        "proposed_corrected_smiles": "F[P-](F)(F)(F)(F)F",
        "chemical_identity": "hexafluorophosphate",
        "correction_basis": (
            "六氟磷酸根原始写法缺少形式负电荷"
        ),
    },
    {
        "ion_role": "anion",
        "raw_smiles": "P(F)(F)(F)(F)(F)F",
        "proposed_corrected_smiles": "F[P-](F)(F)(F)(F)F",
        "chemical_identity": "hexafluorophosphate",
        "correction_basis": (
            "六氟磷酸根原始写法缺少形式负电荷"
        ),
    },
    {
        "ion_role": "anion",
        "raw_smiles": "[CL-]",
        "proposed_corrected_smiles": "[Cl-]",
        "chemical_identity": "chloride",
        "correction_basis": (
            "氯元素符号大小写错误"
        ),
    },
    {
        "ion_role": "anion",
        "raw_smiles": "[BR-]",
        "proposed_corrected_smiles": "[Br-]",
        "chemical_identity": "bromide",
        "correction_basis": (
            "溴元素符号大小写错误"
        ),
    },
    {
        "ion_role": "anion",
        "raw_smiles": "B(C#N)(C#N)(C#N)C#N",
        "proposed_corrected_smiles": "[B-](C#N)(C#N)(C#N)C#N",
        "chemical_identity": "tetracyanoborate",
        "correction_basis": (
            "四氰合硼酸根原始写法缺少形式负电荷"
        ),
    },
]

# PubChem给出的六氟磷酸根规范SMILES为
# F[P-](F)(F)(F)(F)F；四氟硼酸根通常写为
# [B-](F)(F)(F)F。
# 本Cell仍会通过RDKit逐条验证，不直接写回数据库。


def validate_corrected_smiles(smiles: str) -> dict:
    """验证建议修正后的SMILES是否能被RDKit正常解析。"""

    result = {
        "rdkit_parse_ok": False,
        "canonical_smiles": None,
        "formal_charge": None,
        "atom_count": None,
        "validation_error": None,
    }

    try:
        mol = Chem.MolFromSmiles(
            smiles,
            sanitize=True,
        )

        if mol is None:
            result["validation_error"] = (
                "MolFromSmiles returned None"
            )
            return result

        canonical = Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=True,
        )

        formal_charge = sum(
            atom.GetFormalCharge()
            for atom in mol.GetAtoms()
        )

        result.update({
            "rdkit_parse_ok": True,
            "canonical_smiles": canonical,
            "formal_charge": int(formal_charge),
            "atom_count": int(mol.GetNumAtoms()),
        })

    except Exception as error:
        result["validation_error"] = (
            f"{type(error).__name__}: {error}"
        )

    return result


correction_audit_rows = []

for rule in HIGH_CONFIDENCE_CORRECTIONS:
    validation = validate_corrected_smiles(
        rule["proposed_corrected_smiles"]
    )

    source_summary = (
        invalid_unique_summary_df.loc[
            (
                invalid_unique_summary_df[
                    "ion_role"
                ].eq(rule["ion_role"])
            )
            & (
                invalid_unique_summary_df[
                    "raw_smiles"
                ].eq(rule["raw_smiles"])
            )
        ]
    )

    if len(source_summary) != 1:
        raise RuntimeError(
            "高置信度修正规则无法与唯一异常SMILES汇总表"
            f"一一对应：{rule['raw_smiles']}"
        )

    source_row = source_summary.iloc[0]

    correction_audit_rows.append({
        **rule,
        "record_count": int(
            source_row["record_count"]
        ),
        "unique_counterion_count": int(
            source_row[
                "unique_counterion_count"
            ]
        ),
        **validation,
        "expected_ion_charge": -1,
        "charge_check_passed": bool(
            validation["rdkit_parse_ok"]
            and validation["formal_charge"] == -1
        ),
        "correction_decision": (
            "approved_high_confidence"
            if (
                validation["rdkit_parse_ok"]
                and validation["formal_charge"] == -1
            )
            else "validation_failed"
        ),
        "original_database_modified": False,
    })


high_confidence_audit_df = pd.DataFrame(
    correction_audit_rows
)

if not (
    high_confidence_audit_df[
        "rdkit_parse_ok"
    ].all()
    and high_confidence_audit_df[
        "charge_check_passed"
    ].all()
):
    display(high_confidence_audit_df)

    raise RuntimeError(
        "至少一条高置信度修正规则未通过RDKit或电荷审计。"
    )


# ============================================================
# 3. 保留无法直接修正的结构
# ============================================================

approved_raw_smiles = set(
    high_confidence_audit_df[
        "raw_smiles"
    ]
)

unresolved_smiles_df = (
    invalid_unique_summary_df.loc[
        ~invalid_unique_summary_df[
            "raw_smiles"
        ].isin(approved_raw_smiles)
    ]
    .copy()
    .reset_index(drop=True)
)

unresolved_smiles_df[
    "correction_decision"
] = "requires_source_review"

unresolved_smiles_df[
    "reason_for_source_review"
] = ""

reason_map = {
    "CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1": (
        "原始连接关系和价态均存在问题，需核对阴离子名称"
    ),
    "BRC=1C=CC2=C(N=C[N-]2)C1": (
        "元素大小写、环结构和取代位置均需来源确认"
    ),
    "C1=CC(=O)NC1=[O-]": (
        "负电荷和羰基位置不能仅由错误SMILES确定"
    ),
    "CCC[NH3+]C(C)C": (
        "显式氢数导致氮价态异常，需确认阳离子名称"
    ),
}

unresolved_smiles_df[
    "reason_for_source_review"
] = (
    unresolved_smiles_df[
        "raw_smiles"
    ].map(reason_map)
)

if unresolved_smiles_df[
    "reason_for_source_review"
].isna().any():
    missing_reason_smiles = (
        unresolved_smiles_df.loc[
            unresolved_smiles_df[
                "reason_for_source_review"
            ].isna(),
            "raw_smiles",
        ]
        .tolist()
    )

    raise RuntimeError(
        "以下未决SMILES没有配置追溯原因："
        + str(missing_reason_smiles)
    )


# ============================================================
# 4. 读取来源XLS并建立工作表清单
# ============================================================

source_book = xlrd.open_workbook(
    str(CO2_SOURCE_XLS_PATH),
    on_demand=True,
)

sheet_inventory_rows = []

for sheet_index, sheet_name in enumerate(
    source_book.sheet_names()
):
    sheet = source_book.sheet_by_index(
        sheet_index
    )

    sheet_inventory_rows.append({
        "sheet_index": sheet_index,
        "sheet_name": sheet_name,
        "nrows": int(sheet.nrows),
        "ncols": int(sheet.ncols),
        "visibility": int(
            source_book.sheet_visibility[
                sheet_index
            ]
        )
        if hasattr(
            source_book,
            "sheet_visibility",
        )
        else None,
    })

source_sheet_inventory_df = pd.DataFrame(
    sheet_inventory_rows
)


# ============================================================
# 5. 扫描原始和建议修正SMILES的精确文本命中
# ============================================================

search_terms = []

for _, row in invalid_unique_summary_df.iterrows():
    search_terms.append({
        "ion_role": row["ion_role"],
        "search_term": str(
            row["raw_smiles"]
        ).strip(),
        "term_type": "raw_invalid_smiles",
    })

for _, row in high_confidence_audit_df.iterrows():
    search_terms.append({
        "ion_role": row["ion_role"],
        "search_term": str(
            row[
                "proposed_corrected_smiles"
            ]
        ).strip(),
        "term_type": "proposed_corrected_smiles",
    })

search_terms_df = (
    pd.DataFrame(search_terms)
    .drop_duplicates()
    .reset_index(drop=True)
)

search_lookup = {}

for _, row in search_terms_df.iterrows():
    key = row["search_term"]

    search_lookup.setdefault(
        key,
        [],
    ).append({
        "ion_role": row["ion_role"],
        "term_type": row["term_type"],
    })


def row_values_as_json(
    sheet,
    row_index: int,
) -> str:
    if row_index < 0 or row_index >= sheet.nrows:
        return "[]"

    values = [
        sheet.cell_value(
            row_index,
            column_index,
        )
        for column_index in range(
            sheet.ncols
        )
    ]

    return json.dumps(
        values,
        ensure_ascii=False,
        default=str,
    )


exact_hit_rows = []

for sheet_index, sheet_name in enumerate(
    source_book.sheet_names()
):
    sheet = source_book.sheet_by_index(
        sheet_index
    )

    for row_index in range(sheet.nrows):
        for column_index in range(
            sheet.ncols
        ):
            cell_value = sheet.cell_value(
                row_index,
                column_index,
            )

            if not isinstance(
                cell_value,
                str,
            ):
                continue

            normalized_cell_value = (
                cell_value.strip()
            )

            if normalized_cell_value not in search_lookup:
                continue

            for term_info in search_lookup[
                normalized_cell_value
            ]:
                exact_hit_rows.append({
                    "sheet_index": sheet_index,
                    "sheet_name": sheet_name,
                    "excel_row_number": (
                        row_index + 1
                    ),
                    "excel_column_number": (
                        column_index + 1
                    ),
                    "cell_value": (
                        normalized_cell_value
                    ),
                    "ion_role": (
                        term_info["ion_role"]
                    ),
                    "term_type": (
                        term_info["term_type"]
                    ),
                    "row_minus_2_json": (
                        row_values_as_json(
                            sheet,
                            row_index - 2,
                        )
                    ),
                    "row_minus_1_json": (
                        row_values_as_json(
                            sheet,
                            row_index - 1,
                        )
                    ),
                    "matched_row_json": (
                        row_values_as_json(
                            sheet,
                            row_index,
                        )
                    ),
                    "row_plus_1_json": (
                        row_values_as_json(
                            sheet,
                            row_index + 1,
                        )
                    ),
                })

source_xls_exact_hits_df = pd.DataFrame(
    exact_hit_rows
)

if source_xls_exact_hits_df.empty:
    source_xls_exact_hits_df = pd.DataFrame(
        columns=[
            "sheet_index",
            "sheet_name",
            "excel_row_number",
            "excel_column_number",
            "cell_value",
            "ion_role",
            "term_type",
            "row_minus_2_json",
            "row_minus_1_json",
            "matched_row_json",
            "row_plus_1_json",
        ]
    )


# ============================================================
# 6. 搜索可能包含数据表头的行
# ============================================================

HEADER_KEYWORDS = {
    "cation",
    "anion",
    "temperature",
    "pressure",
    "solubility",
    "fugacity",
    "reference",
    "references",
    "experimental method",
    "il-co2",
    "t /k",
    "p /bar",
}

header_candidate_rows = []

for sheet_index, sheet_name in enumerate(
    source_book.sheet_names()
):
    sheet = source_book.sheet_by_index(
        sheet_index
    )

    scan_row_count = min(
        sheet.nrows,
        30,
    )

    for row_index in range(
        scan_row_count
    ):
        row_values = [
            sheet.cell_value(
                row_index,
                column_index,
            )
            for column_index in range(
                sheet.ncols
            )
        ]

        normalized_row_text = " | ".join(
            str(value).strip().lower()
            for value in row_values
            if str(value).strip()
        )

        matched_keywords = sorted(
            keyword
            for keyword in HEADER_KEYWORDS
            if keyword in normalized_row_text
        )

        if len(matched_keywords) >= 2:
            header_candidate_rows.append({
                "sheet_index": sheet_index,
                "sheet_name": sheet_name,
                "excel_row_number": (
                    row_index + 1
                ),
                "matched_keyword_count": (
                    len(matched_keywords)
                ),
                "matched_keywords": ";".join(
                    matched_keywords
                ),
                "row_values_json": json.dumps(
                    row_values,
                    ensure_ascii=False,
                    default=str,
                ),
            })

source_header_candidates_df = pd.DataFrame(
    header_candidate_rows
)

if source_header_candidates_df.empty:
    source_header_candidates_df = pd.DataFrame(
        columns=[
            "sheet_index",
            "sheet_name",
            "excel_row_number",
            "matched_keyword_count",
            "matched_keywords",
            "row_values_json",
        ]
    )


# ============================================================
# 7. 保存结果
# ============================================================

save_csv_atomic(
    high_confidence_audit_df,
    HIGH_CONFIDENCE_AUDIT_PATH,
)

save_csv_atomic(
    unresolved_smiles_df,
    UNRESOLVED_SMILES_PATH,
)

save_csv_atomic(
    source_sheet_inventory_df,
    SOURCE_XLS_SHEET_INVENTORY_PATH,
)

save_csv_atomic(
    source_xls_exact_hits_df,
    SOURCE_XLS_EXACT_HITS_PATH,
)

save_csv_atomic(
    source_header_candidates_df,
    SOURCE_XLS_HEADER_CANDIDATES_PATH,
)


cell7_manifest = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "run_code_version": (
        "03_CO2_invalid_SMILES_source_trace_v1_20260805"
    ),
    "status": (
        "high_confidence_corrections_validated_"
        "and_source_workbook_scanned"
    ),
    "counts": {
        "high_confidence_raw_SMILES": int(
            len(high_confidence_audit_df)
        ),
        "high_confidence_records_covered": int(
            high_confidence_audit_df[
                "record_count"
            ].sum()
        ),
        "unresolved_unique_SMILES": int(
            len(unresolved_smiles_df)
        ),
        "unresolved_records": int(
            unresolved_smiles_df[
                "record_count"
            ].sum()
        ),
        "source_workbook_sheets": int(
            len(source_sheet_inventory_df)
        ),
        "exact_text_hits": int(
            len(source_xls_exact_hits_df)
        ),
        "header_candidate_rows": int(
            len(source_header_candidates_df)
        ),
    },
    "verification": {
        "all_high_confidence_SMILES_parse": bool(
            high_confidence_audit_df[
                "rdkit_parse_ok"
            ].all()
        ),
        "all_high_confidence_charges_equal_minus_one": bool(
            high_confidence_audit_df[
                "charge_check_passed"
            ].all()
        ),
        "original_CO2_database_modified": False,
        "corrected_database_generated": False,
        "exact_matching_rerun": False,
    },
    "outputs": {
        "high_confidence_correction_audit": {
            "path": str(
                HIGH_CONFIDENCE_AUDIT_PATH
            ),
            "sha256": sha256_file(
                HIGH_CONFIDENCE_AUDIT_PATH
            ),
        },
        "unresolved_SMILES": {
            "path": str(
                UNRESOLVED_SMILES_PATH
            ),
            "sha256": sha256_file(
                UNRESOLVED_SMILES_PATH
            ),
        },
        "source_sheet_inventory": {
            "path": str(
                SOURCE_XLS_SHEET_INVENTORY_PATH
            ),
            "sha256": sha256_file(
                SOURCE_XLS_SHEET_INVENTORY_PATH
            ),
        },
        "source_exact_hits": {
            "path": str(
                SOURCE_XLS_EXACT_HITS_PATH
            ),
            "sha256": sha256_file(
                SOURCE_XLS_EXACT_HITS_PATH
            ),
        },
        "source_header_candidates": {
            "path": str(
                SOURCE_XLS_HEADER_CANDIDATES_PATH
            ),
            "sha256": sha256_file(
                SOURCE_XLS_HEADER_CANDIDATES_PATH
            ),
        },
    },
}

save_json_atomic(
    cell7_manifest,
    CELL7_MANIFEST_PATH,
)

source_book.release_resources()


# ============================================================
# 8. 显示结果
# ============================================================

print("=" * 100)
print("Cell 7 高置信度修正规则与来源工作簿追溯完成 ✓")
print("=" * 100)

print(
    "high-confidence raw SMILES       =",
    cell7_manifest[
        "counts"
    ][
        "high_confidence_raw_SMILES"
    ],
)

print(
    "records covered by corrections  =",
    cell7_manifest[
        "counts"
    ][
        "high_confidence_records_covered"
    ],
)

print(
    "unresolved unique SMILES         =",
    cell7_manifest[
        "counts"
    ][
        "unresolved_unique_SMILES"
    ],
)

print(
    "unresolved records               =",
    cell7_manifest[
        "counts"
    ][
        "unresolved_records"
    ],
)

print(
    "source workbook exact text hits  =",
    cell7_manifest[
        "counts"
    ][
        "exact_text_hits"
    ],
)

print("\n高置信度修正规则：")
display(high_confidence_audit_df)

print("\n仍需来源核对的SMILES：")
display(unresolved_smiles_df)

print("\n来源工作簿工作表清单：")
display(source_sheet_inventory_df)

print("\n来源工作簿精确文本命中：")
display(source_xls_exact_hits_df)

print("\n候选表头位置：")
display(source_header_candidates_df)

print("\n输出文件：")
print(HIGH_CONFIDENCE_AUDIT_PATH)
print(UNRESOLVED_SMILES_PATH)
print(SOURCE_XLS_SHEET_INVENTORY_PATH)
print(SOURCE_XLS_EXACT_HITS_PATH)
print(SOURCE_XLS_HEADER_CANDIDATES_PATH)
print(CELL7_MANIFEST_PATH)

Cell 7 高置信度修正规则与来源工作簿追溯完成 ✓
high-confidence raw SMILES       = 6
records covered by corrections  = 2467
unresolved unique SMILES         = 4
unresolved records               = 181
source workbook exact text hits  = 0

高置信度修正规则：


,ion_role,raw_smiles,proposed_corrected_smiles,chemical_identity,correction_basis,record_count,unique_counterion_count,rdkit_parse_ok,canonical_smiles,formal_charge,atom_count,validation_error,expected_ion_charge,charge_check_passed,correction_decision,original_database_modified
0,anion,B(F)(F)(F)F,[B-](F)(F)(F)F,tetrafluoroborate,四配位硼酸根原始写法缺少形式负电荷,953,7,True,F[B-](F)(F)F,-1,5,None,-1,True,approved_high_confidence,False
1,anion,FP(F)(F)(F)(F)F,F[P-](F)(F)(F)(F)F,hexafluorophosphate,六氟磷酸根原始写法缺少形式负电荷,810,5,True,F[P-](F)(F)(F)(F)F,-1,7,None,-1,True,approved_high_confidence,False
2,anion,P(F)(F)(F)(F)(F)F,F[P-](F)(F)(F)(F)F,hexafluorophosphate,六氟磷酸根原始写法缺少形式负电荷,204,2,True,F[P-](F)(F)(F)(F)F,-1,7,None,-1,True,approved_high_confidence,False
3,anion,[CL-],[Cl-],chloride,氯元素符号大小写错误,314,6,True,[Cl-],-1,1,None,-1,True,approved_high_confidence,False
4,anion,[BR-],[Br-],bromide,溴元素符号大小写错误,124,6,True,[Br-],-1,1,None,-1,True,approved_high_confidence,False
5,anion,B(C#N)(C#N)(C#N)C#N,[B-](C#N)(C#N)(C#N)C#N,tetracyanoborate,四氰合硼酸根原始写法缺少形式负电荷,62,1,True,N#C[B-](C#N)(C#N)C#N,-1,9,None,-1,True,approved_high_confidence,False



仍需来源核对的SMILES：


,ion_role,raw_smiles,parse_status,record_count,first_record_id,last_record_id,unique_counterion_count,temperature_min_K,temperature_max_K,pressure_min_bar,pressure_max_bar,solubility_min,solubility_max,correction_decision,reason_for_source_review
0,anion,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,parse_failed,140,7754,7893,1,303.15,363.15,3.90,113.70,0.0971,0.7748,requires_source_review,原始连接关系和价态均存在问题，需核对阴离子名称
1,anion,BRC=1C=CC2=C(N=C[N-]2)C1,parse_failed,26,18889,18984,2,295.15,333.15,0.01,1.02,0.1150,0.4737,requires_source_review,元素大小写、环结构和取代位置均需来源确认
2,anion,C1=CC(=O)NC1=[O-],parse_failed,8,5784,5791,1,333.27,333.27,0.02,3.00,0.0000,0.0258,requires_source_review,负电荷和羰基位置不能仅由错误SMILES确定
3,cation,CCC[NH3+]C(C)C,parse_failed,7,6935,6941,1,304.16,344.84,0.47,0.54,0.0122,0.0238,requires_source_review,显式氢数导致氮价态异常，需确认阳离子名称



来源工作簿工作表清单：


...


来源工作簿精确文本命中：


,sheet_index,sheet_name,excel_row_number,excel_column_number,cell_value,ion_role,term_type,row_minus_2_json,row_minus_1_json,matched_row_json,row_plus_1_json



候选表头位置：


,sheet_index,sheet_name,excel_row_number,matched_keyword_count,matched_keywords,row_values_json
0,2,Table of Contents,3,2,anion;cation,"[""Table S1. Abbreviations, Names and Chemical ..."
1,2,Table of Contents,5,3,pressure;solubility;temperature,"[""Table S2. CO2 Solubility in ILs as a Functio..."
2,2,Table of Contents,7,3,pressure;solubility;temperature,"[""Table S3. SO2 Solubility in ILs as a Functio..."
3,2,Table of Contents,9,3,pressure;solubility;temperature,"[""Table S4. CO Solubility in ILs as a Function..."
4,2,Table of Contents,11,3,pressure;solubility;temperature,"[""Table S5. N2 Solubility in ILs as a Function..."
...,...,...,...,...,...,...
194,20,Table S18,10,2,solubility;temperature,"[""[1,3BMPY][Tf2N]"", ""[1,3EMPY]"", ""[Tf2N]"", ""CO..."
195,20,Table S18,11,2,solubility;temperature,"[""[1,3BMPY][Tf2N]"", ""[1,3EMPY]"", ""[Tf2N]"", ""CO..."
196,20,Table S18,28,2,solubility;temperature,"[""[BMIM][BF4]"", ""[BMIM]"", ""[BF4]"", ""CO2"", 298...."
197,20,Table S18,29,2,solubility;temperature,"[""[BMIM][BF4]"", ""[BMIM]"", ""[BF4]"", ""CO2"", 313...."



输出文件：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\co2_invalid_SMILES_high_confidence_correction_audit.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\co2_invalid_SMILES_unresolved_for_source_review.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\co2_source_workbook_sheet_inventory.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\co2_source_workbook_invalid_SMILES_exact_hits.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\co2_source_workbook_header_candidates.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\logs\co2_invalid_SMILE

In [9]:
# Cell 8 | 基于温度、压力和溶解度追溯未决SMILES的来源信息

from datetime import datetime
import json

import numpy as np
import pandas as pd
import xlrd


# ============================================================
# 1. 输出路径
# ============================================================

UNRESOLVED_RECORDS_PATH = (
    TABLE_DIR
    / "co2_unresolved_SMILES_database_records.csv"
)

SOURCE_TABLE_S2_PATH = (
    TABLE_DIR
    / "co2_source_workbook_TableS2_records.csv"
)

CONDITION_MATCH_PATH = (
    TABLE_DIR
    / "co2_unresolved_SMILES_source_condition_matches.csv"
)

CONDITION_MATCH_SUMMARY_PATH = (
    TABLE_DIR
    / "co2_unresolved_SMILES_source_match_summary.csv"
)

TABLE_S1_ABBREVIATION_HITS_PATH = (
    TABLE_DIR
    / "co2_source_workbook_TableS1_abbreviation_hits.csv"
)

CELL8_MANIFEST_PATH = (
    LOG_DIR
    / "co2_unresolved_SMILES_source_condition_trace_manifest.json"
)


# ============================================================
# 2. 提取CO2数据库中4种未决SMILES的全部记录
# ============================================================

unresolved_record_parts = []

for _, unresolved_row in unresolved_smiles_df.iterrows():
    ion_role = str(
        unresolved_row["ion_role"]
    ).strip()

    raw_smiles = str(
        unresolved_row["raw_smiles"]
    ).strip()

    if ion_role == "cation":
        role_mask = (
            co2_df["cation_smiles"]
            .astype(str)
            .str.strip()
            .eq(raw_smiles)
        )

    elif ion_role == "anion":
        role_mask = (
            co2_df["anion_smiles"]
            .astype(str)
            .str.strip()
            .eq(raw_smiles)
        )

    else:
        raise ValueError(
            f"无法识别的离子角色：{ion_role}"
        )

    role_records = (
        co2_df.loc[
            role_mask,
            [
                "co2_record_id",
                "cation_smiles",
                "anion_smiles",
                "gas_smiles",
                "temperature",
                "pressure",
                "solubility",
            ],
        ]
        .copy()
    )

    role_records.insert(
        1,
        "unresolved_ion_role",
        ion_role,
    )

    role_records.insert(
        2,
        "unresolved_raw_smiles",
        raw_smiles,
    )

    unresolved_record_parts.append(
        role_records
    )


unresolved_database_records_df = pd.concat(
    unresolved_record_parts,
    ignore_index=True,
)

unresolved_database_records_df = (
    unresolved_database_records_df.sort_values(
        by=[
            "unresolved_ion_role",
            "unresolved_raw_smiles",
            "co2_record_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


expected_unresolved_records = int(
    unresolved_smiles_df[
        "record_count"
    ].sum()
)

if len(
    unresolved_database_records_df
) != expected_unresolved_records:
    raise RuntimeError(
        "提取的未决记录数与Cell 7汇总结果不一致："
        f"{len(unresolved_database_records_df)} != "
        f"{expected_unresolved_records}"
    )


# ============================================================
# 3. 读取来源工作簿Table S2
# ============================================================

source_book = xlrd.open_workbook(
    str(CO2_SOURCE_XLS_PATH),
    on_demand=True,
)

table_s2_sheet = source_book.sheet_by_name(
    "Table S2"
)


def normalize_text(value) -> str:
    if value is None:
        return ""

    return str(value).strip()


def safe_float(value):
    if value is None:
        return np.nan

    if isinstance(
        value,
        str,
    ) and not value.strip():
        return np.nan

    try:
        return float(value)

    except (
        TypeError,
        ValueError,
    ):
        return np.nan


# 自动查找Table S2表头，避免硬编码Excel行号
table_s2_header_row_index = None

for row_index in range(
    min(table_s2_sheet.nrows, 20)
):
    row_values = [
        normalize_text(
            table_s2_sheet.cell_value(
                row_index,
                column_index,
            )
        )
        for column_index in range(
            table_s2_sheet.ncols
        )
    ]

    lowered_values = [
        value.lower()
        for value in row_values
    ]

    if (
        any(
            "il-co2 systems" in value
            for value in lowered_values
        )
        and "cation" in lowered_values
        and "anion" in lowered_values
    ):
        table_s2_header_row_index = (
            row_index
        )
        break


if table_s2_header_row_index is None:
    raise RuntimeError(
        "未能在来源工作簿Table S2中识别数据表头。"
    )


table_s2_rows = []

for row_index in range(
    table_s2_header_row_index + 1,
    table_s2_sheet.nrows,
):
    system_value = normalize_text(
        table_s2_sheet.cell_value(
            row_index,
            0,
        )
    )

    cation_abbreviation = normalize_text(
        table_s2_sheet.cell_value(
            row_index,
            1,
        )
    )

    anion_abbreviation = normalize_text(
        table_s2_sheet.cell_value(
            row_index,
            2,
        )
    )

    temperature_value = safe_float(
        table_s2_sheet.cell_value(
            row_index,
            3,
        )
    )

    pressure_value = safe_float(
        table_s2_sheet.cell_value(
            row_index,
            4,
        )
    )

    solubility_value = safe_float(
        table_s2_sheet.cell_value(
            row_index,
            5,
        )
    )

    fugacity_value = safe_float(
        table_s2_sheet.cell_value(
            row_index,
            6,
        )
    )

    experimental_method = normalize_text(
        table_s2_sheet.cell_value(
            row_index,
            13,
        )
    )

    reference_number = normalize_text(
        table_s2_sheet.cell_value(
            row_index,
            14,
        )
    )

    full_reference = normalize_text(
        table_s2_sheet.cell_value(
            row_index,
            15,
        )
    )

    # 略过完全空白的行
    if (
        not system_value
        and not cation_abbreviation
        and not anion_abbreviation
        and np.isnan(temperature_value)
        and np.isnan(pressure_value)
        and np.isnan(solubility_value)
    ):
        continue

    table_s2_rows.append({
        "source_sheet_name": "Table S2",
        "source_excel_row_number": (
            row_index + 1
        ),
        "source_IL_CO2_system": (
            system_value
        ),
        "source_cation_abbreviation": (
            cation_abbreviation
        ),
        "source_anion_abbreviation": (
            anion_abbreviation
        ),
        "source_temperature_K": (
            temperature_value
        ),
        "source_pressure_bar": (
            pressure_value
        ),
        "source_solubility": (
            solubility_value
        ),
        "source_fugacity_bar": (
            fugacity_value
        ),
        "source_experimental_method": (
            experimental_method
        ),
        "source_reference_number": (
            reference_number
        ),
        "source_full_reference": (
            full_reference
        ),
    })


source_table_s2_df = pd.DataFrame(
    table_s2_rows
)

source_table_s2_df = (
    source_table_s2_df.loc[
        source_table_s2_df[
            "source_temperature_K"
        ].notna()
        & source_table_s2_df[
            "source_pressure_bar"
        ].notna()
        & source_table_s2_df[
            "source_solubility"
        ].notna()
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 4. 构建严格数值匹配键
# ============================================================

# 不同文件中的浮点数可能有极小的存储差异，
# 因此使用明确且足够严格的小数位构建诊断匹配键。
TEMPERATURE_KEY_DECIMALS = 2
PRESSURE_KEY_DECIMALS = 5
SOLUBILITY_KEY_DECIMALS = 7


def add_condition_keys(
    dataframe: pd.DataFrame,
    temperature_column: str,
    pressure_column: str,
    solubility_column: str,
) -> pd.DataFrame:
    result = dataframe.copy()

    result[
        "temperature_match_key"
    ] = pd.to_numeric(
        result[temperature_column],
        errors="coerce",
    ).round(
        TEMPERATURE_KEY_DECIMALS
    )

    result[
        "pressure_match_key"
    ] = pd.to_numeric(
        result[pressure_column],
        errors="coerce",
    ).round(
        PRESSURE_KEY_DECIMALS
    )

    result[
        "solubility_match_key"
    ] = pd.to_numeric(
        result[solubility_column],
        errors="coerce",
    ).round(
        SOLUBILITY_KEY_DECIMALS
    )

    return result


unresolved_with_keys_df = add_condition_keys(
    dataframe=unresolved_database_records_df,
    temperature_column="temperature",
    pressure_column="pressure",
    solubility_column="solubility",
)

source_table_s2_with_keys_df = (
    add_condition_keys(
        dataframe=source_table_s2_df,
        temperature_column=(
            "source_temperature_K"
        ),
        pressure_column=(
            "source_pressure_bar"
        ),
        solubility_column=(
            "source_solubility"
        ),
    )
)


# ============================================================
# 5. 根据温度、压力和溶解度匹配来源记录
# ============================================================

condition_matches_df = (
    unresolved_with_keys_df.merge(
        source_table_s2_with_keys_df,
        on=[
            "temperature_match_key",
            "pressure_match_key",
            "solubility_match_key",
        ],
        how="left",
        validate="many_to_many",
        indicator=True,
    )
)


condition_matches_df[
    "source_match_found"
] = (
    condition_matches_df[
        "_merge"
    ].eq("both")
)

condition_matches_df[
    "temperature_difference_K"
] = (
    condition_matches_df[
        "temperature"
    ]
    - condition_matches_df[
        "source_temperature_K"
    ]
).abs()

condition_matches_df[
    "pressure_difference_bar"
] = (
    condition_matches_df[
        "pressure"
    ]
    - condition_matches_df[
        "source_pressure_bar"
    ]
).abs()

condition_matches_df[
    "solubility_difference"
] = (
    condition_matches_df[
        "solubility"
    ]
    - condition_matches_df[
        "source_solubility"
    ]
).abs()


# 统计同一CO2数据库记录对应多少个来源记录
match_count_per_record = (
    condition_matches_df.loc[
        condition_matches_df[
            "source_match_found"
        ]
    ]
    .groupby(
        [
            "unresolved_ion_role",
            "unresolved_raw_smiles",
            "co2_record_id",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": (
                "source_match_count_for_record"
            )
        }
    )
)


condition_matches_df = (
    condition_matches_df.merge(
        match_count_per_record,
        on=[
            "unresolved_ion_role",
            "unresolved_raw_smiles",
            "co2_record_id",
        ],
        how="left",
    )
)

condition_matches_df[
    "source_match_count_for_record"
] = (
    condition_matches_df[
        "source_match_count_for_record"
    ]
    .fillna(0)
    .astype(int)
)

condition_matches_df[
    "source_match_is_unique"
] = (
    condition_matches_df[
        "source_match_count_for_record"
    ].eq(1)
)

condition_matches_df = (
    condition_matches_df.drop(
        columns=["_merge"]
    )
    .sort_values(
        by=[
            "unresolved_ion_role",
            "unresolved_raw_smiles",
            "co2_record_id",
            "source_excel_row_number",
        ],
        kind="stable",
        na_position="last",
    )
    .reset_index(drop=True)
)


# ============================================================
# 6. 按未决SMILES汇总匹配情况
# ============================================================

summary_rows = []

for _, unresolved_row in unresolved_smiles_df.iterrows():
    ion_role = str(
        unresolved_row["ion_role"]
    ).strip()

    raw_smiles = str(
        unresolved_row["raw_smiles"]
    ).strip()

    group_df = (
        condition_matches_df.loc[
            (
                condition_matches_df[
                    "unresolved_ion_role"
                ].eq(ion_role)
            )
            & (
                condition_matches_df[
                    "unresolved_raw_smiles"
                ].eq(raw_smiles)
            )
        ]
        .copy()
    )

    total_database_records = int(
        group_df[
            "co2_record_id"
        ].nunique()
    )

    matched_group_df = (
        group_df.loc[
            group_df[
                "source_match_found"
            ]
        ]
        .copy()
    )

    matched_database_records = int(
        matched_group_df[
            "co2_record_id"
        ].nunique()
    )

    uniquely_matched_records = int(
        matched_group_df.loc[
            matched_group_df[
                "source_match_is_unique"
            ],
            "co2_record_id",
        ].nunique()
    )

    ambiguously_matched_records = int(
        matched_group_df.loc[
            matched_group_df[
                "source_match_count_for_record"
            ].gt(1),
            "co2_record_id",
        ].nunique()
    )

    unmatched_database_records = (
        total_database_records
        - matched_database_records
    )

    def join_unique_values(
        series: pd.Series,
    ) -> str:
        values = sorted({
            str(value).strip()
            for value in series.dropna()
            if str(value).strip()
        })

        return " | ".join(values)

    summary_rows.append({
        "ion_role": ion_role,
        "raw_smiles": raw_smiles,
        "total_database_records": (
            total_database_records
        ),
        "matched_database_records": (
            matched_database_records
        ),
        "uniquely_matched_records": (
            uniquely_matched_records
        ),
        "ambiguously_matched_records": (
            ambiguously_matched_records
        ),
        "unmatched_database_records": (
            unmatched_database_records
        ),
        "matched_record_fraction": (
            matched_database_records
            / total_database_records
            if total_database_records > 0
            else np.nan
        ),
        "source_IL_CO2_systems": (
            join_unique_values(
                matched_group_df[
                    "source_IL_CO2_system"
                ]
            )
        ),
        "source_cation_abbreviations": (
            join_unique_values(
                matched_group_df[
                    "source_cation_abbreviation"
                ]
            )
        ),
        "source_anion_abbreviations": (
            join_unique_values(
                matched_group_df[
                    "source_anion_abbreviation"
                ]
            )
        ),
        "source_experimental_methods": (
            join_unique_values(
                matched_group_df[
                    "source_experimental_method"
                ]
            )
        ),
        "source_reference_numbers": (
            join_unique_values(
                matched_group_df[
                    "source_reference_number"
                ]
            )
        ),
        "source_full_references": (
            join_unique_values(
                matched_group_df[
                    "source_full_reference"
                ]
            )
        ),
    })


source_match_summary_df = pd.DataFrame(
    summary_rows
)


# ============================================================
# 7. 在Table S1中追溯匹配到的离子缩写
# ============================================================

table_s1_sheet = source_book.sheet_by_name(
    "Table S1"
)

abbreviation_targets = set()

for column in [
    "source_cation_abbreviation",
    "source_anion_abbreviation",
]:
    abbreviation_targets.update({
        str(value).strip()
        for value in condition_matches_df.loc[
            condition_matches_df[
                "source_match_found"
            ],
            column,
        ].dropna()
        if str(value).strip()
    })


table_s1_hit_rows = []

for row_index in range(
    table_s1_sheet.nrows
):
    row_values = [
        table_s1_sheet.cell_value(
            row_index,
            column_index,
        )
        for column_index in range(
            table_s1_sheet.ncols
        )
    ]

    normalized_row_values = [
        normalize_text(value)
        for value in row_values
    ]

    for column_index, cell_value in enumerate(
        normalized_row_values
    ):
        if cell_value not in abbreviation_targets:
            continue

        table_s1_hit_rows.append({
            "source_sheet_name": "Table S1",
            "source_excel_row_number": (
                row_index + 1
            ),
            "source_excel_column_number": (
                column_index + 1
            ),
            "matched_abbreviation": (
                cell_value
            ),
            "complete_row_json": json.dumps(
                row_values,
                ensure_ascii=False,
                default=str,
            ),
        })


table_s1_abbreviation_hits_df = pd.DataFrame(
    table_s1_hit_rows
)

if table_s1_abbreviation_hits_df.empty:
    table_s1_abbreviation_hits_df = (
        pd.DataFrame(
            columns=[
                "source_sheet_name",
                "source_excel_row_number",
                "source_excel_column_number",
                "matched_abbreviation",
                "complete_row_json",
            ]
        )
    )


source_book.release_resources()


# ============================================================
# 8. 保存结果
# ============================================================

save_csv_atomic(
    unresolved_database_records_df,
    UNRESOLVED_RECORDS_PATH,
)

save_csv_atomic(
    source_table_s2_df,
    SOURCE_TABLE_S2_PATH,
)

save_csv_atomic(
    condition_matches_df,
    CONDITION_MATCH_PATH,
)

save_csv_atomic(
    source_match_summary_df,
    CONDITION_MATCH_SUMMARY_PATH,
)

save_csv_atomic(
    table_s1_abbreviation_hits_df,
    TABLE_S1_ABBREVIATION_HITS_PATH,
)


cell8_manifest = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "run_code_version": (
        "04_CO2_unresolved_SMILES_condition_trace_v1_20260805"
    ),
    "status": (
        "unresolved_SMILES_source_condition_trace_completed"
    ),
    "matching_method": {
        "fields": [
            "temperature",
            "pressure",
            "experimental_solubility",
        ],
        "temperature_rounding_decimals": (
            TEMPERATURE_KEY_DECIMALS
        ),
        "pressure_rounding_decimals": (
            PRESSURE_KEY_DECIMALS
        ),
        "solubility_rounding_decimals": (
            SOLUBILITY_KEY_DECIMALS
        ),
    },
    "counts": {
        "unresolved_unique_SMILES": int(
            len(unresolved_smiles_df)
        ),
        "unresolved_database_records": int(
            len(unresolved_database_records_df)
        ),
        "source_TableS2_numeric_records": int(
            len(source_table_s2_df)
        ),
        "matched_unresolved_database_records": int(
            condition_matches_df.loc[
                condition_matches_df[
                    "source_match_found"
                ],
                "co2_record_id",
            ].nunique()
        ),
        "uniquely_matched_unresolved_records": int(
            condition_matches_df.loc[
                condition_matches_df[
                    "source_match_is_unique"
                ],
                "co2_record_id",
            ].nunique()
        ),
        "ambiguously_matched_unresolved_records": int(
            condition_matches_df.loc[
                condition_matches_df[
                    "source_match_count_for_record"
                ].gt(1),
                "co2_record_id",
            ].nunique()
        ),
        "TableS1_abbreviation_hits": int(
            len(table_s1_abbreviation_hits_df)
        ),
    },
    "verification": {
        "original_CO2_database_modified": False,
        "source_workbook_modified": False,
        "corrected_database_generated": False,
        "exact_candidate_matching_rerun": False,
    },
    "outputs": {
        "unresolved_database_records": {
            "path": str(
                UNRESOLVED_RECORDS_PATH
            ),
            "sha256": sha256_file(
                UNRESOLVED_RECORDS_PATH
            ),
        },
        "source_TableS2_records": {
            "path": str(
                SOURCE_TABLE_S2_PATH
            ),
            "sha256": sha256_file(
                SOURCE_TABLE_S2_PATH
            ),
        },
        "condition_matches": {
            "path": str(
                CONDITION_MATCH_PATH
            ),
            "sha256": sha256_file(
                CONDITION_MATCH_PATH
            ),
        },
        "condition_match_summary": {
            "path": str(
                CONDITION_MATCH_SUMMARY_PATH
            ),
            "sha256": sha256_file(
                CONDITION_MATCH_SUMMARY_PATH
            ),
        },
        "TableS1_abbreviation_hits": {
            "path": str(
                TABLE_S1_ABBREVIATION_HITS_PATH
            ),
            "sha256": sha256_file(
                TABLE_S1_ABBREVIATION_HITS_PATH
            ),
        },
    },
}

save_json_atomic(
    cell8_manifest,
    CELL8_MANIFEST_PATH,
)


# ============================================================
# 9. 显示结果
# ============================================================

print("=" * 100)
print("Cell 8 未决SMILES来源条件追溯完成 ✓")
print("=" * 100)

print(
    "unresolved database records       =",
    cell8_manifest[
        "counts"
    ][
        "unresolved_database_records"
    ],
)

print(
    "matched unresolved records        =",
    cell8_manifest[
        "counts"
    ][
        "matched_unresolved_database_records"
    ],
)

print(
    "uniquely matched records          =",
    cell8_manifest[
        "counts"
    ][
        "uniquely_matched_unresolved_records"
    ],
)

print(
    "ambiguously matched records       =",
    cell8_manifest[
        "counts"
    ][
        "ambiguously_matched_unresolved_records"
    ],
)

print("\n按未决SMILES汇总的来源追溯结果：")
display(source_match_summary_df)

print("\n成功匹配的来源记录前30行：")
display(
    condition_matches_df.loc[
        condition_matches_df[
            "source_match_found"
        ]
    ].head(30)
)

print("\nTable S1离子缩写命中：")
display(table_s1_abbreviation_hits_df)

print("\n输出文件：")
print(UNRESOLVED_RECORDS_PATH)
print(SOURCE_TABLE_S2_PATH)
print(CONDITION_MATCH_PATH)
print(CONDITION_MATCH_SUMMARY_PATH)
print(TABLE_S1_ABBREVIATION_HITS_PATH)
print(CELL8_MANIFEST_PATH)

Cell 8 未决SMILES来源条件追溯完成 ✓
unresolved database records       = 181
matched unresolved records        = 115
uniquely matched records          = 114
ambiguously matched records       = 1

按未决SMILES汇总的来源追溯结果：


,ion_role,raw_smiles,total_database_records,matched_database_records,uniquely_matched_records,ambiguously_matched_records,unmatched_database_records,matched_record_fraction,source_IL_CO2_systems,source_cation_abbreviations,source_anion_abbreviations,source_experimental_methods,source_reference_numbers,source_full_references
0,anion,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,140,112,112,0,28,0.800,"[P6,6,6,14][C12H25PhSO3]-CO2","[P6,6,6,14]",[C12H25PhSO3],Synthetic (Bubble Point) Method,[199],"Yushu Chen, Fabrice Mutelet, Jean-Noël Jaubert..."
1,anion,BRC=1C=CC2=C(N=C[N-]2)C1,26,0,0,0,26,0.000,,,,,,
2,anion,C1=CC(=O)NC1=[O-],8,3,2,1,5,0.375,[HMIM][SAC]-CO2,[HMIM],[SAC],Gravimetric Microbalance Method,[148],"Mark J. Muldoon, Sudhir N. V. K. Aki, Jessica ..."
3,cation,CCC[NH3+]C(C)C,7,0,0,0,7,0.000,,,,,,



成功匹配的来源记录前30行：


,co2_record_id,unresolved_ion_role,unresolved_raw_smiles,cation_smiles,anion_smiles,gas_smiles,temperature,pressure,solubility,temperature_match_key,...,source_fugacity_bar,source_experimental_method,source_reference_number,source_full_reference,source_match_found,temperature_difference_K,pressure_difference_bar,solubility_difference,source_match_count_for_record,source_match_is_unique
31,5789,anion,C1=CC(=O)NC1=[O-],CCCCCCC[N+](C)C1=CN=CC=C1,C1=CC(=O)NC1=[O-],O=C=O,333.27,1.5,0.0115,333.27,...,1.492303,Gravimetric Microbalance Method,[148],"Mark J. Muldoon, Sudhir N. V. K. Aki, Jessica ...",True,0.0,0.0,0.0,1,True
32,5790,anion,C1=CC(=O)NC1=[O-],CCCCCCC[N+](C)C1=CN=CC=C1,C1=CC(=O)NC1=[O-],O=C=O,333.27,2.0,0.0167,333.27,...,1.986322,Gravimetric Microbalance Method,[148],"Mark J. Muldoon, Sudhir N. V. K. Aki, Jessica ...",True,0.0,0.0,0.0,1,True
33,5791,anion,C1=CC(=O)NC1=[O-],CCCCCCC[N+](C)C1=CN=CC=C1,C1=CC(=O)NC1=[O-],O=C=O,333.27,3.0,0.0258,333.27,...,2.969248,Gravimetric Microbalance Method,[148],"Mark J. Muldoon, Sudhir N. V. K. Aki, Jessica ...",True,0.0,0.0,0.0,2,False
34,5791,anion,C1=CC(=O)NC1=[O-],CCCCCCC[N+](C)C1=CN=CC=C1,C1=CC(=O)NC1=[O-],O=C=O,333.27,3.0,0.0258,333.27,...,2.969248,Gravimetric Microbalance Method,[148],"Mark J. Muldoon, Sudhir N. V. K. Aki, Jessica ...",True,0.0,0.0,0.0,2,False
63,7782,anion,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,CCCCCCCCCCCCCC[N+](C)(CCCCCC)CCCCCC.C(C)(C)P(C...,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,O=C=O,303.15,3.9,0.0971,303.15,...,3.828772,Synthetic (Bubble Point) Method,[199],"Yushu Chen, Fabrice Mutelet, Jean-Noël Jaubert...",True,0.0,0.0,0.0,1,True
64,7783,anion,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,CCCCCCCCCCCCCC[N+](C)(CCCCCC)CCCCCC.C(C)(C)P(C...,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,O=C=O,313.15,4.6,0.0971,313.15,...,4.511063,Synthetic (Bubble Point) Method,[199],"Yushu Chen, Fabrice Mutelet, Jean-Noël Jaubert...",True,0.0,0.0,0.0,1,True
65,7784,anion,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,CCCCCCCCCCCCCC[N+](C)(CCCCCC)CCCCCC.C(C)(C)P(C...,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,O=C=O,323.15,5.7,0.0971,323.15,...,5.577134,Synthetic (Bubble Point) Method,[199],"Yushu Chen, Fabrice Mutelet, Jean-Noël Jaubert...",True,0.0,0.0,0.0,1,True
66,7785,anion,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,CCCCCCCCCCCCCC[N+](C)(CCCCCC)CCCCCC.C(C)(C)P(C...,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,O=C=O,333.15,6.1,0.0971,333.15,...,5.973011,Synthetic (Bubble Point) Method,[199],"Yushu Chen, Fabrice Mutelet, Jean-Noël Jaubert...",True,0.0,0.0,0.0,1,True
67,7786,anion,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,CCCCCCCCCCCCCC[N+](C)(CCCCCC)CCCCCC.C(C)(C)P(C...,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,O=C=O,343.15,6.4,0.0971,343.15,...,6.273558,Synthetic (Bubble Point) Method,[199],"Yushu Chen, Fabrice Mutelet, Jean-Noël Jaubert...",True,0.0,0.0,0.0,1,True
68,7787,anion,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,CCCCCCCCCCCCCC[N+](C)(CCCCCC)CCCCCC.C(C)(C)P(C...,CCCCCCCCCCCS(=O)(=O)[O-]C1=CC=CC=C1,O=C=O,353.15,6.7,0.0971,353.15,...,6.574408,Synthetic (Bubble Point) Method,[199],"Yushu Chen, Fabrice Mutelet, Jean-Noël Jaubert...",True,0.0,0.0,0.0,1,True



Table S1离子缩写命中：


,source_sheet_name,source_excel_row_number,source_excel_column_number,matched_abbreviation,complete_row_json



输出文件：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\co2_unresolved_SMILES_database_records.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\co2_source_workbook_TableS2_records.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\co2_unresolved_SMILES_source_condition_matches.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\co2_unresolved_SMILES_source_match_summary.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\co2_source_workbook_TableS1_abbreviation_hits.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\logs\co2_unresolved_SMILES_sour

In [10]:
# Cell 9 | 使用结构修正后的CO2数据库重新完成最低5%候选精确匹配

CELL9_RUN_CODE_VERSION = (
    "09_lowest5pct_exact_pair_rematch_after_correction_v1_20260806"
)

CORRECTED_CO2_DATABASE_PATH = (
    TABLE_DIR
    / (
        "co2_solubility_structure_"
        "corrected_for_exact_matching.csv"
    )
)

CORRECTION_RECORD_AUDIT_PATH = (
    TABLE_DIR
    / "co2_structure_correction_record_audit.csv"
)

CORRECTION_RULE_AUDIT_PATH = (
    TABLE_DIR
    / "co2_structure_correction_rule_audit.csv"
)


# ============================================================
# 1. 检查结构修正后的数据库
# ============================================================

if not CORRECTED_CO2_DATABASE_PATH.exists():
    raise FileNotFoundError(
        "没有找到结构修正后的CO2数据库。\n"
        "请先依次运行Cell 6、Cell 7和Cell 8。\n"
        f"预期文件：\n{CORRECTED_CO2_DATABASE_PATH}"
    )

corrected_co2_source_df = pd.read_csv(
    CORRECTED_CO2_DATABASE_PATH,
    encoding="utf-8-sig",
    low_memory=False,
)

co2_records_after_df = prepare_co2_records(
    corrected_co2_source_df,
    use_corrected_columns=True,
)


# ============================================================
# 2. 最低5%的854种候选精确匹配
# ============================================================

lowest5_exact_after_df = exact_pair_merge(
    lowest5_candidates_df,
    co2_records_after_df,
)

lowest5_exact_summary_after_df = (
    build_exact_match_summary(
        lowest5_exact_after_df
    )
)

lowest5_exact_T_window_after_df = (
    lowest5_exact_after_df.loc[
        lowest5_exact_after_df[
            "within_29815K_window"
        ]
    ]
    .copy()
    .sort_values(
        [
            "rank_viscosity_nonhalide",
            "absolute_temperature_difference_to_29815_K",
            "absolute_pressure_difference_to_101325_bar",
            "co2_record_id",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

lowest5_exact_scope_after_df = pd.DataFrame([
    {
        "scope": (
            "Lowest5pct_nonhalide"
        ),
        "rank_limit": (
            EXPECTED_LOWEST5_COUNT
        ),
        "total_candidate_count": int(
            len(
                lowest5_candidates_df
            )
        ),
        "exact_matched_candidate_count": int(
            lowest5_exact_after_df[
                "candidate_pair_key_canonical"
            ].nunique()
        ),
        "exact_matched_candidate_fraction": (
            float(
                lowest5_exact_after_df[
                    "candidate_pair_key_canonical"
                ].nunique()
            )
            / float(
                len(
                    lowest5_candidates_df
                )
            )
        ),
        "matched_experimental_record_count": int(
            len(
                lowest5_exact_after_df
            )
        ),
        "matched_records_within_29815K_window": int(
            len(
                lowest5_exact_T_window_after_df
            )
        ),
        "matched_records_created_by_structure_correction": int(
            lowest5_exact_after_df[
                "structure_correction_applied"
            ].sum()
        ),
    }
])


# ============================================================
# 3. 全部17082种非简单卤素候选仅作内部审计
# ============================================================

all_nonhalide_candidates_df = (
    prepare_candidate_table(
        ALL_NONHALIDE_CANDIDATE_PATH,
        expected_count=(
            EXPECTED_ALL_NONHALIDE_COUNT
        ),
        expected_max_rank=(
            EXPECTED_ALL_NONHALIDE_COUNT
        ),
    )
)

all_nonhalide_exact_after_df = (
    exact_pair_merge(
        all_nonhalide_candidates_df,
        co2_records_after_df,
    )
)

all_nonhalide_exact_summary_after_df = (
    build_exact_match_summary(
        all_nonhalide_exact_after_df
    )
)


# ============================================================
# 4. 修正前后对照
# ============================================================

lowest5_before_after_comparison_df = (
    pd.DataFrame([
        {
            "scope": (
                "Lowest5pct_nonhalide"
            ),
            "candidate_count": (
                EXPECTED_LOWEST5_COUNT
            ),
            "before_exact_matched_candidates": int(
                lowest5_exact_before_df[
                    "candidate_pair_key_canonical"
                ].nunique()
            ),
            "before_matched_records": int(
                len(
                    lowest5_exact_before_df
                )
            ),
            "before_records_T29815_window": int(
                len(
                    lowest5_exact_T_window_before_df
                )
            ),
            "after_exact_matched_candidates": int(
                lowest5_exact_after_df[
                    "candidate_pair_key_canonical"
                ].nunique()
            ),
            "after_matched_records": int(
                len(
                    lowest5_exact_after_df
                )
            ),
            "after_records_T29815_window": int(
                len(
                    lowest5_exact_T_window_after_df
                )
            ),
            "new_exact_matched_candidates": int(
                lowest5_exact_after_df[
                    "candidate_pair_key_canonical"
                ].nunique()
                - lowest5_exact_before_df[
                    "candidate_pair_key_canonical"
                ].nunique()
            ),
            "new_matched_records": int(
                len(
                    lowest5_exact_after_df
                )
                - len(
                    lowest5_exact_before_df
                )
            ),
            "new_records_T29815_window": int(
                len(
                    lowest5_exact_T_window_after_df
                )
                - len(
                    lowest5_exact_T_window_before_df
                )
            ),
        }
    ])
)


# ============================================================
# 5. 输出路径
# ============================================================

LOWEST5_EXACT_RECORDS_AFTER_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "exact_pair_matched_records_after_correction.csv"
    )
)

LOWEST5_EXACT_SUMMARY_AFTER_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "exact_pair_match_summary_after_correction.csv"
    )
)

LOWEST5_EXACT_SCOPE_AFTER_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "exact_match_counts_after_correction.csv"
    )
)

LOWEST5_EXACT_T_WINDOW_AFTER_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "exact_match_T29815_window_after_correction.csv"
    )
)

LOWEST5_BEFORE_AFTER_COMPARISON_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "exact_match_before_after_comparison.csv"
    )
)

ALL_NONHALIDE_EXACT_AUDIT_PATH = (
    TABLE_DIR
    / (
        "all_nonhalide_candidate_CO2_"
        "exact_pair_match_internal_audit.csv"
    )
)

LOWEST5_EXACT_AFTER_MANIFEST_PATH = (
    LOG_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "exact_rematch_after_correction_manifest.json"
    )
)


# ============================================================
# 6. 保存
# ============================================================

save_csv_atomic(
    lowest5_exact_after_df,
    LOWEST5_EXACT_RECORDS_AFTER_PATH,
)

save_csv_atomic(
    lowest5_exact_summary_after_df,
    LOWEST5_EXACT_SUMMARY_AFTER_PATH,
)

save_csv_atomic(
    lowest5_exact_scope_after_df,
    LOWEST5_EXACT_SCOPE_AFTER_PATH,
)

save_csv_atomic(
    lowest5_exact_T_window_after_df,
    LOWEST5_EXACT_T_WINDOW_AFTER_PATH,
)

save_csv_atomic(
    lowest5_before_after_comparison_df,
    LOWEST5_BEFORE_AFTER_COMPARISON_PATH,
)

save_csv_atomic(
    all_nonhalide_exact_summary_after_df,
    ALL_NONHALIDE_EXACT_AUDIT_PATH,
)


# ============================================================
# 7. 结构修正统计
# ============================================================

if CORRECTION_RECORD_AUDIT_PATH.exists():
    correction_record_audit_df = (
        pd.read_csv(
            CORRECTION_RECORD_AUDIT_PATH,
            encoding="utf-8-sig",
            low_memory=False,
        )
    )

    corrected_record_count = int(
        len(
            correction_record_audit_df
        )
    )

else:
    correction_record_audit_df = (
        pd.DataFrame()
    )

    corrected_record_count = int(
        co2_records_after_df[
            "structure_correction_applied"
        ].sum()
    )


# ============================================================
# 8. Manifest
# ============================================================

lowest5_exact_after_manifest = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "run_code_version": (
        CELL9_RUN_CODE_VERSION
    ),
    "status": (
        "lowest5pct_exact_pair_matching_"
        "after_structure_correction_completed"
    ),
    "analysis_scope": {
        "formal_candidate_scope": (
            "lowest_5pct_non_simple_halide_candidates"
        ),
        "formal_candidate_count": int(
            len(
                lowest5_candidates_df
            )
        ),
        "internal_audit_scope": (
            "all_non_simple_halide_candidates"
        ),
        "internal_audit_candidate_count": int(
            len(
                all_nonhalide_candidates_df
            )
        ),
        "comparison_basis": (
            "exact_canonical_complete_ion_pair_identity"
        ),
        "temperature_window_center_K": (
            TARGET_T_K
        ),
        "temperature_window_half_width_K": (
            TEMPERATURE_WINDOW_K
        ),
        "molecular_fingerprint_used": False,
        "similarity_threshold_applied": False,
        "Henry_constant_calculated": False,
        "different_temperatures_combined": False,
        "different_pressures_combined": False,
    },
    "counts": {
        "corrected_database_rows": int(
            len(
                corrected_co2_source_df
            )
        ),
        "valid_complete_pair_records_after_correction": int(
            co2_records_after_df[
                "corrected_pair_key_canonical"
            ].notna().sum()
        ),
        "records_with_structure_correction": (
            corrected_record_count
        ),
        "lowest5pct_exact_matched_candidates": int(
            lowest5_exact_after_df[
                "candidate_pair_key_canonical"
            ].nunique()
        ),
        "lowest5pct_matched_experimental_records": int(
            len(
                lowest5_exact_after_df
            )
        ),
        "lowest5pct_records_within_29815K_window": int(
            len(
                lowest5_exact_T_window_after_df
            )
        ),
        "all_nonhalide_exact_matched_candidates": int(
            all_nonhalide_exact_after_df[
                "candidate_pair_key_canonical"
            ].nunique()
        ),
        "all_nonhalide_matched_experimental_records": int(
            len(
                all_nonhalide_exact_after_df
            )
        ),
    },
    "verification": {
        "formal_candidate_count_is_854": bool(
            len(
                lowest5_candidates_df
            )
            == EXPECTED_LOWEST5_COUNT
        ),
        "formal_exact_match_count_is_zero": bool(
            lowest5_exact_after_df[
                "candidate_pair_key_canonical"
            ].nunique()
            == 0
        ),
        "all_nonhalide_candidate_count_is_17082": bool(
            len(
                all_nonhalide_candidates_df
            )
            == EXPECTED_ALL_NONHALIDE_COUNT
        ),
        "all_nonhalide_exact_match_count_is_14": bool(
            all_nonhalide_exact_after_df[
                "candidate_pair_key_canonical"
            ].nunique()
            == 14
        ),
        "original_CO2_database_modified": False,
        "corrected_CO2_database_modified": False,
        "different_temperatures_combined": False,
        "different_pressures_combined": False,
    },
    "inputs": {
        "lowest5pct_candidate_table": str(
            LOWEST5_CANDIDATE_PATH
        ),
        "all_nonhalide_candidate_table": str(
            ALL_NONHALIDE_CANDIDATE_PATH
        ),
        "corrected_CO2_database": str(
            CORRECTED_CO2_DATABASE_PATH
        ),
        "correction_rule_audit": str(
            CORRECTION_RULE_AUDIT_PATH
        ),
        "correction_record_audit": str(
            CORRECTION_RECORD_AUDIT_PATH
        ),
    },
    "outputs": {
        "lowest5pct_matched_records": str(
            LOWEST5_EXACT_RECORDS_AFTER_PATH
        ),
        "lowest5pct_match_summary": str(
            LOWEST5_EXACT_SUMMARY_AFTER_PATH
        ),
        "lowest5pct_scope_summary": str(
            LOWEST5_EXACT_SCOPE_AFTER_PATH
        ),
        "lowest5pct_T29815_records": str(
            LOWEST5_EXACT_T_WINDOW_AFTER_PATH
        ),
        "lowest5pct_before_after_comparison": str(
            LOWEST5_BEFORE_AFTER_COMPARISON_PATH
        ),
        "all_nonhalide_internal_audit": str(
            ALL_NONHALIDE_EXACT_AUDIT_PATH
        ),
    },
}

save_json_atomic(
    lowest5_exact_after_manifest,
    LOWEST5_EXACT_AFTER_MANIFEST_PATH,
)


# ============================================================
# 9. 正式审计
# ============================================================

if (
    lowest5_exact_after_df[
        "candidate_pair_key_canonical"
    ].nunique()
    != 0
):
    raise RuntimeError(
        "结构修正后的854种最低5%候选中"
        "出现了精确匹配，与当前正式结果不一致。"
    )

if (
    all_nonhalide_exact_after_df[
        "candidate_pair_key_canonical"
    ].nunique()
    != 14
):
    raise RuntimeError(
        "全部17082种非简单卤素候选的"
        "精确匹配数量不是14，"
        "请检查数据库修正结果。"
    )


# ============================================================
# 10. 输出
# ============================================================

print(
    "=" * 104
)

print(
    "Cell 9 最低5%候选结构修正后"
    "完整离子对精确匹配完成 ✓"
)

print(
    "=" * 104
)

print(
    "lowest 5% candidates                    =",
    len(
        lowest5_candidates_df
    ),
)

print(
    "lowest 5% exact matched candidates       =",
    lowest5_exact_after_df[
        "candidate_pair_key_canonical"
    ].nunique(),
)

print(
    "lowest 5% matched experimental records   =",
    len(
        lowest5_exact_after_df
    ),
)

print(
    "all nonhalide exact matched candidates   =",
    all_nonhalide_exact_after_df[
        "candidate_pair_key_canonical"
    ].nunique(),
)

print(
    "all nonhalide matched records            =",
    len(
        all_nonhalide_exact_after_df
    ),
)

print(
    "formal match summary                     =",
    LOWEST5_EXACT_SUMMARY_AFTER_PATH,
)

print(
    "internal complete-scope audit             =",
    ALL_NONHALIDE_EXACT_AUDIT_PATH,
)

print(
    "manifest                                  =",
    LOWEST5_EXACT_AFTER_MANIFEST_PATH,
)

display(
    lowest5_exact_scope_after_df
)

display(
    all_nonhalide_exact_summary_after_df
)

Cell 9 最低5%候选结构修正后完整离子对精确匹配完成 ✓
lowest 5% candidates                    = 854
lowest 5% exact matched candidates       = 0
lowest 5% matched experimental records   = 0
all nonhalide exact matched candidates   = 14
all nonhalide matched records            = 502
formal match summary                     = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\lowest5pct_candidate_CO2_exact_pair_match_summary_after_correction.csv
internal complete-scope audit             = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\all_nonhalide_candidate_CO2_exact_pair_match_internal_audit.csv
manifest                                  = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\logs\lowest5pct_candidate_CO2_exact_rematch_after_correction_manifest.json


,scope,rank_limit,total_candidate_count,exact_matched_candidate_count,exact_matched_candidate_fraction,matched_experimental_record_count,matched_records_within_29815K_window,matched_records_created_by_structure_correction
0,Lowest5pct_nonhalide,854,854,0,0.0,0,0,0


,candidate_pair_key_canonical,rank_viscosity_nonhalide,candidate_cation_smiles,candidate_anion_smiles,pred_lg_eta,pred_eta_mPa_s,n_CO2_records,n_unique_temperatures,temperature_min_K,temperature_max_K,pressure_min_bar,pressure_max_bar,solubility_min,solubility_max,n_corrected_source_records,n_records_within_29815K_window
0,[NH3+]CCO||O=C[O-],1594,[NH3+]CCO,O=C[O-],1.995028,98.861662,48,3,303.00,323.00,0.4400,10.01000,0.0202,0.3083,0,0
1,[NH3+]CCO||CC(=O)[O-],3073,[NH3+]CCO,CC(=O)[O-],2.213417,163.461937,66,6,298.15,328.15,0.7600,15.42000,0.0081,0.4860,0,6
2,CCCC[P+](CCCC)(CCCC)CCCC||O=C[O-],3751,CCCC[P+](CCCC)(CCCC)CCCC,O=C[O-],2.292756,196.225683,9,3,298.00,298.20,0.1000,20.00000,0.0080,0.3480,0,9
3,CCCCCCCCCCCCCC[P+](CCCCCC)(CCCCCC)CCCCCC||CCC(...,5951,CCCCCCCCCCCCCC[P+](CCCCCC)(CCCCCC)CCCCCC,CCC(=O)[O-],2.514554,327.004793,88,6,283.15,373.15,0.0000,1.22000,0.0291,0.4924,0,0
4,CCCCCCCCCCCCCC[P+](CCCCCC)(CCCCCC)CCCCCC||N#C[...,6293,CCCCCCCCCCCCCC[P+](CCCCCC)(CCCCCC)CCCCCC,N#C[N-]C#N,2.547086,352.440802,125,99,271.11,363.40,3.0400,902.48000,0.1110,0.8430,0,14
5,CCCC[P+](CCCC)(CCCC)CCCN||NCC(=O)[O-],6842,CCCC[P+](CCCC)(CCCC)CCCN,NCC(=O)[O-],2.593378,392.082681,1,1,296.15,296.15,1.0000,1.00000,0.5238,0.5238,0,0
6,C[NH+](CCO)CCO||CC(=O)[O-],7307,C[NH+](CCO)CCO,CC(=O)[O-],2.632785,429.323860,18,3,298.15,328.15,1.2400,15.42000,0.0024,0.0761,0,6
7,OCC[NH2+]CCO||CC(=O)[O-],7506,OCC[NH2+]CCO,CC(=O)[O-],2.647177,443.789435,18,3,298.15,328.15,1.2500,15.15000,0.0074,0.1076,0,6
8,CCCCCCCCCCCCCC[P+](CCCCCC)(CCCCCC)CCCCCC||NCC(...,10012,CCCCCCCCCCCCCC[P+](CCCCCC)(CCCCCC)CCCCCC,NCC(=O)[O-],2.851378,710.195176,14,1,295.15,295.15,0.0100,1.30000,0.4329,0.5291,0,0
9,CC(C)C[P+](C)(CC(C)C)CC(C)C||COS(=O)(=O)[O-],10188,CC(C)C[P+](C)(CC(C)C)CC(C)C,COS(=O)(=O)[O-],2.866842,735.939432,29,24,313.16,363.30,6.5900,126.40000,0.0700,0.4550,0,0


In [11]:
# Cell 10 | 分析854种最低5%候选与CO2数据库中共享离子的相关完整离子液体

CELL10_RUN_CODE_VERSION = (
    "10_lowest5pct_shared_ion_related_complete_IL_v1_20260806"
)


# ============================================================
# 1. 构建CO2数据库唯一完整离子液体库
# ============================================================

experimental_records_valid_df = (
    co2_records_after_df.loc[
        co2_records_after_df[
            "corrected_pair_key_canonical"
        ].notna()
    ]
    .copy()
)

experimental_pair_library_df = (
    experimental_records_valid_df.groupby(
        [
            "corrected_pair_key_canonical",
            "corrected_cation_smiles_canonical",
            "corrected_anion_smiles_canonical",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_CO2_records=(
            "co2_record_id",
            "size",
        ),
        n_unique_temperatures=(
            "co2_temperature_K",
            "nunique",
        ),
        temperature_min_K=(
            "co2_temperature_K",
            "min",
        ),
        temperature_max_K=(
            "co2_temperature_K",
            "max",
        ),
        n_unique_pressures=(
            "co2_pressure_bar",
            "nunique",
        ),
        pressure_min_bar=(
            "co2_pressure_bar",
            "min",
        ),
        pressure_max_bar=(
            "co2_pressure_bar",
            "max",
        ),
        solubility_min=(
            "co2_solubility",
            "min",
        ),
        solubility_max=(
            "co2_solubility",
            "max",
        ),
        n_records_within_29815K_window=(
            "within_29815K_window",
            "sum",
        ),
        n_structure_corrected_records=(
            "structure_correction_applied",
            "sum",
        ),
    )
    .rename(
        columns={
            "corrected_pair_key_canonical": (
                "experimental_pair_key_canonical"
            ),
            "corrected_cation_smiles_canonical": (
                "experimental_cation_smiles"
            ),
            "corrected_anion_smiles_canonical": (
                "experimental_anion_smiles"
            ),
        }
    )
    .sort_values(
        [
            "n_records_within_29815K_window",
            "n_CO2_records",
            "experimental_pair_key_canonical",
        ],
        ascending=[
            False,
            False,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 2. 建立实验离子集合
# ============================================================

experimental_cation_set = set(
    experimental_pair_library_df[
        "experimental_cation_smiles"
    ].dropna()
)

experimental_anion_set = set(
    experimental_pair_library_df[
        "experimental_anion_smiles"
    ].dropna()
)

experimental_pair_set = set(
    experimental_pair_library_df[
        "experimental_pair_key_canonical"
    ].dropna()
)


# ============================================================
# 3. 完整阳离子相同、阴离子不同
# ============================================================

candidate_link_columns = [
    "candidate_record_id",
    "rank_viscosity_nonhalide",
    "virtual_IL_key",
    "candidate_pair_key_canonical",
    "candidate_cation_smiles_original",
    "candidate_anion_smiles_original",
    "candidate_cation_smiles",
    "candidate_anion_smiles",
    "pred_lg_eta",
    "pred_eta_mPa_s",
    "T_pred_K",
    "P_pred_bar",
]

candidate_link_base_df = (
    lowest5_candidates_df[
        candidate_link_columns
    ]
    .copy()
)

same_cation_links_df = (
    candidate_link_base_df.merge(
        experimental_pair_library_df,
        left_on=(
            "candidate_cation_smiles"
        ),
        right_on=(
            "experimental_cation_smiles"
        ),
        how="inner",
        validate="many_to_many",
    )
)

same_cation_links_df = (
    same_cation_links_df.loc[
        same_cation_links_df[
            "candidate_pair_key_canonical"
        ]
        != same_cation_links_df[
            "experimental_pair_key_canonical"
        ]
    ]
    .copy()
)

same_cation_links_df[
    "same_cation"
] = True

same_cation_links_df[
    "same_anion"
] = False

same_cation_links_df[
    "same_complete_ion_pair"
] = False

same_cation_links_df[
    "shared_ion_count"
] = 1

same_cation_links_df[
    "shared_ion_relation"
] = (
    "same_cation_different_anion"
)


# ============================================================
# 4. 完整阴离子相同、阳离子不同
# ============================================================

same_anion_links_df = (
    candidate_link_base_df.merge(
        experimental_pair_library_df,
        left_on=(
            "candidate_anion_smiles"
        ),
        right_on=(
            "experimental_anion_smiles"
        ),
        how="inner",
        validate="many_to_many",
    )
)

same_anion_links_df = (
    same_anion_links_df.loc[
        same_anion_links_df[
            "candidate_pair_key_canonical"
        ]
        != same_anion_links_df[
            "experimental_pair_key_canonical"
        ]
    ]
    .copy()
)

same_anion_links_df[
    "same_cation"
] = False

same_anion_links_df[
    "same_anion"
] = True

same_anion_links_df[
    "same_complete_ion_pair"
] = False

same_anion_links_df[
    "shared_ion_count"
] = 1

same_anion_links_df[
    "shared_ion_relation"
] = (
    "different_cation_same_anion"
)


# ============================================================
# 5. 合并所有共享离子相关完整离子液体
# ============================================================

related_complete_IL_pairs_df = pd.concat(
    [
        same_cation_links_df,
        same_anion_links_df,
    ],
    ignore_index=True,
)

if not related_complete_IL_pairs_df.empty:
    related_complete_IL_pairs_df = (
        related_complete_IL_pairs_df.drop_duplicates(
            subset=[
                "candidate_pair_key_canonical",
                "experimental_pair_key_canonical",
                "shared_ion_relation",
            ]
        )
        .sort_values(
            [
                "rank_viscosity_nonhalide",
                "n_records_within_29815K_window",
                "n_CO2_records",
                "experimental_pair_key_canonical",
            ],
            ascending=[
                True,
                False,
                False,
                True,
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 6. 构建854种候选逐条证据汇总
# ============================================================

candidate_evidence_summary_df = (
    lowest5_candidates_df.copy()
)

candidate_evidence_summary_df[
    "exact_complete_pair_in_CO2_database"
] = candidate_evidence_summary_df[
    "candidate_pair_key_canonical"
].isin(
    experimental_pair_set
)

candidate_evidence_summary_df[
    "candidate_cation_in_CO2_database"
] = candidate_evidence_summary_df[
    "candidate_cation_smiles"
].isin(
    experimental_cation_set
)

candidate_evidence_summary_df[
    "candidate_anion_in_CO2_database"
] = candidate_evidence_summary_df[
    "candidate_anion_smiles"
].isin(
    experimental_anion_set
)


def classify_ion_evidence(row):
    if row[
        "exact_complete_pair_in_CO2_database"
    ]:
        return "exact_complete_pair"

    if (
        row[
            "candidate_cation_in_CO2_database"
        ]
        and row[
            "candidate_anion_in_CO2_database"
        ]
    ):
        return (
            "both_ions_individually_observed"
        )

    if row[
        "candidate_cation_in_CO2_database"
    ]:
        return "cation_only_observed"

    if row[
        "candidate_anion_in_CO2_database"
    ]:
        return "anion_only_observed"

    return "neither_ion_observed"


candidate_evidence_summary_df[
    "ion_evidence_class"
] = candidate_evidence_summary_df.apply(
    classify_ion_evidence,
    axis=1,
)


# ============================================================
# 7. 统计每种候选的相关完整离子液体覆盖
# ============================================================

if related_complete_IL_pairs_df.empty:
    shared_pair_counts_df = pd.DataFrame(
        columns=[
            "candidate_pair_key_canonical",
            "n_shared_complete_IL_pairs",
            "n_same_cation_different_anion_pairs",
            "n_different_cation_same_anion_pairs",
            "total_CO2_records_via_shared_ion",
            "total_records_within_29815K_window",
            "total_unique_temperature_counts",
        ]
    )

else:
    shared_pair_counts_df = (
        related_complete_IL_pairs_df.groupby(
            "candidate_pair_key_canonical",
            as_index=False,
        )
        .agg(
            n_shared_complete_IL_pairs=(
                "experimental_pair_key_canonical",
                "nunique",
            ),
            total_CO2_records_via_shared_ion=(
                "n_CO2_records",
                "sum",
            ),
            total_records_within_29815K_window=(
                "n_records_within_29815K_window",
                "sum",
            ),
            total_unique_temperature_counts=(
                "n_unique_temperatures",
                "sum",
            ),
        )
    )

    relation_counts_df = (
        pd.crosstab(
            related_complete_IL_pairs_df[
                "candidate_pair_key_canonical"
            ],
            related_complete_IL_pairs_df[
                "shared_ion_relation"
            ],
        )
        .reset_index()
    )

    for relation_column in [
        "same_cation_different_anion",
        "different_cation_same_anion",
    ]:
        if (
            relation_column
            not in relation_counts_df.columns
        ):
            relation_counts_df[
                relation_column
            ] = 0

    relation_counts_df = (
        relation_counts_df.rename(
            columns={
                "same_cation_different_anion": (
                    "n_same_cation_different_anion_pairs"
                ),
                "different_cation_same_anion": (
                    "n_different_cation_same_anion_pairs"
                ),
            }
        )
    )

    shared_pair_counts_df = (
        shared_pair_counts_df.merge(
            relation_counts_df[
                [
                    "candidate_pair_key_canonical",
                    "n_same_cation_different_anion_pairs",
                    "n_different_cation_same_anion_pairs",
                ]
            ],
            on=(
                "candidate_pair_key_canonical"
            ),
            how="left",
            validate="one_to_one",
        )
    )

candidate_evidence_summary_df = (
    candidate_evidence_summary_df.merge(
        shared_pair_counts_df,
        on=(
            "candidate_pair_key_canonical"
        ),
        how="left",
        validate="one_to_one",
    )
)

count_columns = [
    "n_shared_complete_IL_pairs",
    "n_same_cation_different_anion_pairs",
    "n_different_cation_same_anion_pairs",
    "total_CO2_records_via_shared_ion",
    "total_records_within_29815K_window",
    "total_unique_temperature_counts",
]

for column in count_columns:
    if (
        column
        not in candidate_evidence_summary_df.columns
    ):
        candidate_evidence_summary_df[
            column
        ] = 0

    candidate_evidence_summary_df[
        column
    ] = (
        candidate_evidence_summary_df[
            column
        ]
        .fillna(0)
        .astype(int)
    )

candidate_evidence_summary_df[
    "n_exact_complete_pair_matches"
] = (
    candidate_evidence_summary_df[
        "exact_complete_pair_in_CO2_database"
    ].astype(int)
)

candidate_evidence_summary_df = (
    candidate_evidence_summary_df.sort_values(
        [
            "rank_viscosity_nonhalide",
            "candidate_pair_key_canonical",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 8. 关联实验记录展开
# ============================================================

if related_complete_IL_pairs_df.empty:
    shared_ion_experimental_records_df = (
        pd.DataFrame()
    )

else:
    link_identification_columns = [
        "candidate_record_id",
        "rank_viscosity_nonhalide",
        "virtual_IL_key",
        "candidate_pair_key_canonical",
        "candidate_cation_smiles",
        "candidate_anion_smiles",
        "pred_lg_eta",
        "pred_eta_mPa_s",
        "T_pred_K",
        "P_pred_bar",
        "experimental_pair_key_canonical",
        "experimental_cation_smiles",
        "experimental_anion_smiles",
        "same_cation",
        "same_anion",
        "same_complete_ion_pair",
        "shared_ion_count",
        "shared_ion_relation",
    ]

    shared_ion_experimental_records_df = (
        related_complete_IL_pairs_df[
            link_identification_columns
        ]
        .merge(
            experimental_records_valid_df,
            left_on=(
                "experimental_pair_key_canonical"
            ),
            right_on=(
                "corrected_pair_key_canonical"
            ),
            how="inner",
            validate="many_to_many",
        )
    )

    shared_ion_experimental_records_df[
        "absolute_temperature_difference_to_candidate_K"
    ] = (
        shared_ion_experimental_records_df[
            "co2_temperature_K"
        ]
        - shared_ion_experimental_records_df[
            "T_pred_K"
        ]
    ).abs()

    shared_ion_experimental_records_df[
        "absolute_pressure_difference_to_candidate_bar"
    ] = (
        shared_ion_experimental_records_df[
            "co2_pressure_bar"
        ]
        - shared_ion_experimental_records_df[
            "P_pred_bar"
        ]
    ).abs()

    shared_ion_experimental_records_df[
        "within_candidate_temperature_window"
    ] = (
        shared_ion_experimental_records_df[
            "absolute_temperature_difference_to_candidate_K"
        ]
        <= TEMPERATURE_WINDOW_K
    )

    shared_ion_experimental_records_df = (
        shared_ion_experimental_records_df.sort_values(
            [
                "rank_viscosity_nonhalide",
                "absolute_temperature_difference_to_candidate_K",
                "absolute_pressure_difference_to_candidate_bar",
                "experimental_pair_key_canonical",
                "co2_record_id",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

condition_matched_records_df = (
    shared_ion_experimental_records_df.loc[
        shared_ion_experimental_records_df[
            "within_candidate_temperature_window"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
    if not shared_ion_experimental_records_df.empty
    else pd.DataFrame(
        columns=(
            shared_ion_experimental_records_df.columns
        )
    )
)


# ============================================================
# 9. 每种候选选择覆盖最充分的相关完整离子液体
# ============================================================

if related_complete_IL_pairs_df.empty:
    best_coverage_pairs_df = pd.DataFrame(
        columns=(
            related_complete_IL_pairs_df.columns
        )
    )

else:
    best_coverage_pairs_df = (
        related_complete_IL_pairs_df.sort_values(
            [
                "rank_viscosity_nonhalide",
                "n_records_within_29815K_window",
                "n_CO2_records",
                "n_unique_temperatures",
                "experimental_pair_key_canonical",
            ],
            ascending=[
                True,
                False,
                False,
                False,
                True,
            ],
            kind="mergesort",
        )
        .drop_duplicates(
            subset=[
                "candidate_pair_key_canonical"
            ],
            keep="first",
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 10. 选择温度和压力最接近预测条件的代表性实验记录
# ============================================================

if (
    best_coverage_pairs_df.empty
    or shared_ion_experimental_records_df.empty
):
    representative_comparison_df = (
        pd.DataFrame()
    )

else:
    selected_pair_keys_df = (
        best_coverage_pairs_df[
            [
                "candidate_pair_key_canonical",
                "experimental_pair_key_canonical",
            ]
        ]
        .copy()
    )

    representative_record_pool_df = (
        shared_ion_experimental_records_df.merge(
            selected_pair_keys_df,
            on=[
                "candidate_pair_key_canonical",
                "experimental_pair_key_canonical",
            ],
            how="inner",
            validate="many_to_one",
        )
    )

    representative_records_df = (
        representative_record_pool_df.sort_values(
            [
                "rank_viscosity_nonhalide",
                "absolute_temperature_difference_to_candidate_K",
                "absolute_pressure_difference_to_candidate_bar",
                "co2_record_id",
            ],
            kind="mergesort",
        )
        .drop_duplicates(
            subset=[
                "candidate_pair_key_canonical"
            ],
            keep="first",
        )
        .reset_index(
            drop=True
        )
    )

    representative_record_columns = (
        representative_records_df[
            [
                "candidate_pair_key_canonical",
                "experimental_pair_key_canonical",
                "co2_record_id",
                "co2_temperature_K",
                "co2_pressure_bar",
                "co2_solubility",
                "absolute_temperature_difference_to_candidate_K",
                "absolute_pressure_difference_to_candidate_bar",
                "within_candidate_temperature_window",
                "structure_correction_applied",
                "applied_correction_rule_ids",
            ]
        ]
        .rename(
            columns={
                "co2_record_id": (
                    "representative_CO2_record_id"
                ),
                "co2_temperature_K": (
                    "representative_temperature_K"
                ),
                "co2_pressure_bar": (
                    "representative_pressure_bar"
                ),
                "co2_solubility": (
                    "representative_CO2_solubility"
                ),
                "absolute_temperature_difference_to_candidate_K": (
                    "representative_temperature_difference_K"
                ),
                "absolute_pressure_difference_to_candidate_bar": (
                    "representative_pressure_difference_bar"
                ),
                "within_candidate_temperature_window": (
                    "representative_within_29815K_window"
                ),
                "structure_correction_applied": (
                    "representative_structure_correction_applied"
                ),
                "applied_correction_rule_ids": (
                    "representative_correction_rule_ids"
                ),
            }
        )
    )

    representative_comparison_df = (
        best_coverage_pairs_df.merge(
            representative_record_columns,
            on=[
                "candidate_pair_key_canonical",
                "experimental_pair_key_canonical",
            ],
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            [
                "rank_viscosity_nonhalide",
                "experimental_pair_key_canonical",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    representative_comparison_df[
        "comparison_basis"
    ] = representative_comparison_df[
        "shared_ion_relation"
    ]


# ============================================================
# 11. 最低5%证据范围汇总
# ============================================================

evidence_scope_summary_df = pd.DataFrame([
    {
        "scope": (
            "Lowest5pct_nonhalide"
        ),
        "total_candidates": int(
            len(
                candidate_evidence_summary_df
            )
        ),
        "exact_complete_pair_candidates": int(
            (
                candidate_evidence_summary_df[
                    "ion_evidence_class"
                ]
                == "exact_complete_pair"
            ).sum()
        ),
        "both_ions_individually_observed_candidates": int(
            (
                candidate_evidence_summary_df[
                    "ion_evidence_class"
                ]
                == (
                    "both_ions_individually_observed"
                )
            ).sum()
        ),
        "cation_only_observed_candidates": int(
            (
                candidate_evidence_summary_df[
                    "ion_evidence_class"
                ]
                == "cation_only_observed"
            ).sum()
        ),
        "anion_only_observed_candidates": int(
            (
                candidate_evidence_summary_df[
                    "ion_evidence_class"
                ]
                == "anion_only_observed"
            ).sum()
        ),
        "neither_ion_observed_candidates": int(
            (
                candidate_evidence_summary_df[
                    "ion_evidence_class"
                ]
                == "neither_ion_observed"
            ).sum()
        ),
        "candidates_with_any_shared_ion": int(
            (
                candidate_evidence_summary_df[
                    "n_shared_complete_IL_pairs"
                ]
                > 0
            ).sum()
        ),
        "related_candidate_experimental_pair_links": int(
            len(
                related_complete_IL_pairs_df
            )
        ),
        "related_experimental_records": int(
            len(
                shared_ion_experimental_records_df
            )
        ),
        "records_within_29815K_window": int(
            len(
                condition_matched_records_df
            )
        ),
        "representative_candidate_comparisons": int(
            len(
                representative_comparison_df
            )
        ),
    }
])


# ============================================================
# 12. 输出路径
# ============================================================

EXPERIMENTAL_PAIR_LIBRARY_PATH = (
    TABLE_DIR
    / (
        "co2_unique_complete_IL_pair_"
        "library_after_correction.csv"
    )
)

LOWEST5_SHARED_ION_SUMMARY_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "shared_ion_summary.csv"
    )
)

LOWEST5_RELATED_COMPLETE_PAIRS_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "related_complete_IL_pairs.csv"
    )
)

LOWEST5_SHARED_EXPERIMENTAL_RECORDS_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "shared_ion_experimental_records.csv"
    )
)

LOWEST5_CONDITION_MATCHED_RECORDS_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "condition_matched_records.csv"
    )
)

LOWEST5_BEST_COVERAGE_PAIRS_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "best_coverage_pairs.csv"
    )
)

LOWEST5_REPRESENTATIVE_COMPARISON_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "representative_comparison.csv"
    )
)

LOWEST5_EVIDENCE_SCOPE_SUMMARY_PATH = (
    TABLE_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "evidence_scope_summary.csv"
    )
)

LOWEST5_SHARED_ION_MANIFEST_PATH = (
    LOG_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "shared_ion_evidence_manifest.json"
    )
)


# ============================================================
# 13. 保存结果
# ============================================================

save_csv_atomic(
    experimental_pair_library_df,
    EXPERIMENTAL_PAIR_LIBRARY_PATH,
)

save_csv_atomic(
    candidate_evidence_summary_df,
    LOWEST5_SHARED_ION_SUMMARY_PATH,
)

save_csv_atomic(
    related_complete_IL_pairs_df,
    LOWEST5_RELATED_COMPLETE_PAIRS_PATH,
)

save_csv_atomic(
    shared_ion_experimental_records_df,
    LOWEST5_SHARED_EXPERIMENTAL_RECORDS_PATH,
)

save_csv_atomic(
    condition_matched_records_df,
    LOWEST5_CONDITION_MATCHED_RECORDS_PATH,
)

save_csv_atomic(
    best_coverage_pairs_df,
    LOWEST5_BEST_COVERAGE_PAIRS_PATH,
)

save_csv_atomic(
    representative_comparison_df,
    LOWEST5_REPRESENTATIVE_COMPARISON_PATH,
)

save_csv_atomic(
    evidence_scope_summary_df,
    LOWEST5_EVIDENCE_SCOPE_SUMMARY_PATH,
)


# ============================================================
# 14. 保存Manifest
# ============================================================

lowest5_shared_ion_manifest = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "run_code_version": (
        CELL10_RUN_CODE_VERSION
    ),
    "status": (
        "lowest5pct_shared_ion_related_"
        "complete_IL_analysis_completed"
    ),
    "analysis_scope": {
        "candidate_scope": (
            "lowest_5pct_non_simple_halide_candidates"
        ),
        "candidate_count": int(
            len(
                lowest5_candidates_df
            )
        ),
        "comparison_basis": [
            "exact_canonical_complete_ion_pair_identity",
            "exact_canonical_cation_identity",
            "exact_canonical_anion_identity",
        ],
        "related_complete_IL_definition": (
            "same_complete_cation_or_same_complete_anion"
        ),
        "representative_pair_selection": [
            (
                "records_within_29815K_window_descending"
            ),
            "total_CO2_records_descending",
            "unique_temperature_count_descending",
            "experimental_pair_key_ascending",
        ],
        "representative_record_selection": [
            (
                "absolute_temperature_difference_"
                "to_29815K_ascending"
            ),
            (
                "absolute_pressure_difference_"
                "to_101325bar_ascending"
            ),
            "record_id_ascending",
        ],
        "molecular_fingerprint_used": False,
        "similarity_threshold_applied": False,
        "Henry_constant_calculated": False,
        "different_temperatures_combined": False,
        "different_pressures_combined": False,
        "solubility_values_averaged": False,
    },
    "counts": evidence_scope_summary_df.iloc[
        0
    ].to_dict(),
    "verification": {
        "candidate_count_is_854": bool(
            len(
                candidate_evidence_summary_df
            )
            == EXPECTED_LOWEST5_COUNT
        ),
        "candidate_pair_keys_unique": bool(
            not candidate_evidence_summary_df[
                "candidate_pair_key_canonical"
            ].duplicated().any()
        ),
        "experimental_pair_keys_unique": bool(
            not experimental_pair_library_df[
                "experimental_pair_key_canonical"
            ].duplicated().any()
        ),
        "formal_exact_complete_pair_count_is_zero": bool(
            (
                candidate_evidence_summary_df[
                    "ion_evidence_class"
                ]
                == "exact_complete_pair"
            ).sum()
            == 0
        ),
        "molecular_fingerprint_used": False,
        "similarity_threshold_applied": False,
        "original_CO2_database_modified": False,
        "corrected_CO2_database_modified": False,
        "different_temperatures_combined": False,
        "different_pressures_combined": False,
    },
    "outputs": {
        "experimental_pair_library": str(
            EXPERIMENTAL_PAIR_LIBRARY_PATH
        ),
        "candidate_shared_ion_summary": str(
            LOWEST5_SHARED_ION_SUMMARY_PATH
        ),
        "related_complete_IL_pairs": str(
            LOWEST5_RELATED_COMPLETE_PAIRS_PATH
        ),
        "shared_ion_experimental_records": str(
            LOWEST5_SHARED_EXPERIMENTAL_RECORDS_PATH
        ),
        "condition_matched_records": str(
            LOWEST5_CONDITION_MATCHED_RECORDS_PATH
        ),
        "best_coverage_pairs": str(
            LOWEST5_BEST_COVERAGE_PAIRS_PATH
        ),
        "representative_comparison": str(
            LOWEST5_REPRESENTATIVE_COMPARISON_PATH
        ),
        "evidence_scope_summary": str(
            LOWEST5_EVIDENCE_SCOPE_SUMMARY_PATH
        ),
    },
}

save_json_atomic(
    lowest5_shared_ion_manifest,
    LOWEST5_SHARED_ION_MANIFEST_PATH,
)


# ============================================================
# 15. 输出结果
# ============================================================

print(
    "=" * 108
)

print(
    "Cell 10 最低5%候选与CO2数据库"
    "共享离子相关完整离子液体分析完成 ✓"
)

print(
    "=" * 108
)

print(
    "formal candidates                         =",
    len(
        candidate_evidence_summary_df
    ),
)

print(
    "experimental unique complete IL pairs     =",
    len(
        experimental_pair_library_df
    ),
)

print(
    "exact complete-pair candidates             =",
    int(
        (
            candidate_evidence_summary_df[
                "ion_evidence_class"
            ]
            == "exact_complete_pair"
        ).sum()
    ),
)

print(
    "candidates with any shared ion             =",
    int(
        (
            candidate_evidence_summary_df[
                "n_shared_complete_IL_pairs"
            ]
            > 0
        ).sum()
    ),
)

print(
    "candidate-experimental pair links          =",
    len(
        related_complete_IL_pairs_df
    ),
)

print(
    "shared-ion experimental records            =",
    len(
        shared_ion_experimental_records_df
    ),
)

print(
    "records within 298.15±0.50 K               =",
    len(
        condition_matched_records_df
    ),
)

print(
    "representative comparisons                 =",
    len(
        representative_comparison_df
    ),
)

print(
    "candidate evidence summary                 =",
    LOWEST5_SHARED_ION_SUMMARY_PATH,
)

print(
    "related complete IL pairs                  =",
    LOWEST5_RELATED_COMPLETE_PAIRS_PATH,
)

print(
    "representative comparison                  =",
    LOWEST5_REPRESENTATIVE_COMPARISON_PATH,
)

print(
    "manifest                                   =",
    LOWEST5_SHARED_ION_MANIFEST_PATH,
)

display(
    evidence_scope_summary_df
)

display(
    representative_comparison_df.head(
        30
    )
)

Cell 10 最低5%候选与CO2数据库共享离子相关完整离子液体分析完成 ✓
formal candidates                         = 854
experimental unique complete IL pairs     = 343
exact complete-pair candidates             = 0
candidates with any shared ion             = 427
candidate-experimental pair links          = 2138
shared-ion experimental records            = 145858
records within 298.15±0.50 K               = 9596
representative comparisons                 = 427
candidate evidence summary                 = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\lowest5pct_candidate_CO2_shared_ion_summary.csv
related complete IL pairs                  = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\tables\lowest5pct_candidate_CO2_related_complete_IL_pairs.csv
representative comparison                  = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_vis

,scope,total_candidates,exact_complete_pair_candidates,both_ions_individually_observed_candidates,cation_only_observed_candidates,anion_only_observed_candidates,neither_ion_observed_candidates,candidates_with_any_shared_ion,related_candidate_experimental_pair_links,related_experimental_records,records_within_29815K_window,representative_candidate_comparisons
0,Lowest5pct_nonhalide,854,0,26,26,375,427,427,2138,145858,9596,427


,candidate_record_id,rank_viscosity_nonhalide,virtual_IL_key,candidate_pair_key_canonical,candidate_cation_smiles_original,candidate_anion_smiles_original,candidate_cation_smiles,candidate_anion_smiles,pred_lg_eta,pred_eta_mPa_s,...,representative_CO2_record_id,representative_temperature_K,representative_pressure_bar,representative_CO2_solubility,representative_temperature_difference_K,representative_pressure_difference_bar,representative_within_29815K_window,representative_structure_correction_applied,representative_correction_rule_ids,comparison_basis
0,4,4,C[NH+]1CCCC1=O||NCC(=O)[O-],C[NH+]1CCCC1=O||NCC(=O)[O-],C[NH+]1CCCC1=O,NCC(=O)[O-],C[NH+]1CCCC1=O,NCC(=O)[O-],0.911188,8.150576,...,8102,295.15,0.90,0.4425,3.00,0.11325,False,False,,different_cation_same_anion
1,7,7,CC[NH+](C)C||O=C[O-],CC[NH+](C)C||O=C[O-],CC[NH+](C)C,O=C[O-],CC[NH+](C)C,O=C[O-],0.949522,8.902705,...,7698,298.10,1.00,0.0340,0.05,0.01325,True,False,,different_cation_same_anion
2,10,10,C[NH+]1CCCC1=O||O=C([O-])C(F)(F)F,C[NH+]1CCCC1=O||O=C([O-])C(F)(F)F,C[NH+]1CCCC1=O,O=C([O-])C(F)(F)F,C[NH+]1CCCC1=O,O=C([O-])C(F)(F)F,0.978778,9.523087,...,2429,298.17,13.30,0.2080,0.02,12.28675,True,False,,different_cation_same_anion
3,11,11,C[NH+]1CCCC1=O||O=[N+]([O-])[O-],C[NH+]1CCCC1=O||O=[N+]([O-])[O-],C[NH+]1CCCC1=O,O=[N+]([O-])[O-],C[NH+]1CCCC1=O,O=[N+]([O-])[O-],0.983600,9.629418,...,1114,298.20,10.31,0.0992,0.05,9.29675,True,False,,different_cation_same_anion
4,13,13,C[NH+]1CCCC1=O||N#C[B-](C#N)(C#N)C#N,C[NH+]1CCCC1=O||N#C[B-](C#N)(C#N)C#N,C[NH+]1CCCC1=O,N#C[B-](C#N)(C#N)C#N,C[NH+]1CCCC1=O,N#C[B-](C#N)(C#N)C#N,1.008470,10.196939,...,14447,293.86,23.72,0.5007,4.29,22.70675,False,False,,different_cation_same_anion
5,18,18,CC[NH+](CC)CC||O=C[O-],CC[NH+](CC)CC||O=C[O-],CC[NH+](CC)CC,O=C[O-],CC[NH+](CC)CC,O=C[O-],1.040571,10.979201,...,7698,298.10,1.00,0.0340,0.05,0.01325,True,False,,different_cation_same_anion
6,32,32,Cc1cccc[nH+]1||CCC(=O)[O-],Cc1cccc[nH+]1||CCC(=O)[O-],Cc1cccc[nH+]1,CCC(=O)[O-],Cc1cccc[nH+]1,CCC(=O)[O-],1.104099,12.708638,...,10497,298.10,0.50,0.1970,0.05,0.51325,True,False,,different_cation_same_anion
7,35,35,CCn1cc[nH+]c1||CCC(=O)[O-],CCn1cc[nH+]c1||CCC(=O)[O-],CCn1cc[nH+]c1,CCC(=O)[O-],CCn1cc[nH+]c1,CCC(=O)[O-],1.110030,12.883398,...,10497,298.10,0.50,0.1970,0.05,0.51325,True,False,,different_cation_same_anion
8,44,44,C[NH+]1CCCC1=O||N#C[C-](C#N)C#N,C[NH+]1CCCC1=O||N#C[C-](C#N)C#N,C[NH+]1CCCC1=O,N#C[C-](C#N)C#N,C[NH+]1CCCC1=O,N#C[C-](C#N)C#N,1.152312,14.200783,...,18623,298.16,32.05,0.4000,0.01,31.03675,True,False,,different_cation_same_anion
9,47,47,C1CC[NH2+]C1||O=C[O-],C1CC[NH2+]C1||O=C[O-],C1CC[NH2+]C1,O=C[O-],C1CC[NH2+]C1,O=C[O-],1.156243,14.329879,...,7698,298.10,1.00,0.0340,0.05,0.01325,True,False,,different_cation_same_anion


In [12]:
# Cell 11 | 完成最低5%候选与CO2数据库对照结果的最终审计

CELL11_RUN_CODE_VERSION = (
    "11_lowest5pct_CO2_crosscheck_final_audit_v1_20260806"
)


# ============================================================
# 1. 清理已经废弃的Top20、Top50和Top100结果
# ============================================================

deprecated_paths = [
    TABLE_DIR
    / "top100_candidate_shared_ion_match_all_pairs.csv",

    TABLE_DIR
    / "top100_candidate_shared_ion_summary.csv",

    TABLE_DIR
    / "top_candidate_shared_ion_scope_summary.csv",

    TABLE_DIR
    / "top100_candidate_shared_ion_best_coverage_pairs.csv",

    TABLE_DIR
    / "top100_candidate_shared_ion_experimental_records.csv",

    LOG_DIR
    / "top100_candidate_shared_ion_evidence_manifest.json",

    TABLE_DIR
    / "candidate_CO2_exact_match_counts_by_scope_after_correction.csv",
]

deprecated_cleanup_records = []

for deprecated_path in deprecated_paths:
    existed_before_cleanup = (
        deprecated_path.exists()
    )

    if existed_before_cleanup:
        deprecated_path.unlink()

    deprecated_cleanup_records.append({
        "file_name": (
            deprecated_path.name
        ),
        "path": str(
            deprecated_path
        ),
        "existed_before_cleanup": bool(
            existed_before_cleanup
        ),
        "exists_after_cleanup": bool(
            deprecated_path.exists()
        ),
        "removed_successfully": bool(
            not deprecated_path.exists()
        ),
    })

deprecated_cleanup_df = pd.DataFrame(
    deprecated_cleanup_records
)

DEPRECATED_CLEANUP_AUDIT_PATH = (
    TABLE_DIR
    / (
        "deprecated_top_candidate_"
        "output_cleanup_audit.csv"
    )
)

save_csv_atomic(
    deprecated_cleanup_df,
    DEPRECATED_CLEANUP_AUDIT_PATH,
)


# ============================================================
# 2. 检查必要文件
# ============================================================

required_output_paths = {
    "exact_records_before": (
        LOWEST5_EXACT_RECORDS_BEFORE_PATH
    ),
    "exact_summary_before": (
        LOWEST5_EXACT_SUMMARY_BEFORE_PATH
    ),
    "exact_counts_before": (
        LOWEST5_EXACT_SCOPE_BEFORE_PATH
    ),
    "exact_T_window_before": (
        LOWEST5_EXACT_T_WINDOW_BEFORE_PATH
    ),
    "exact_before_manifest": (
        LOWEST5_EXACT_BEFORE_MANIFEST_PATH
    ),
    "exact_records_after": (
        LOWEST5_EXACT_RECORDS_AFTER_PATH
    ),
    "exact_summary_after": (
        LOWEST5_EXACT_SUMMARY_AFTER_PATH
    ),
    "exact_counts_after": (
        LOWEST5_EXACT_SCOPE_AFTER_PATH
    ),
    "exact_T_window_after": (
        LOWEST5_EXACT_T_WINDOW_AFTER_PATH
    ),
    "before_after_comparison": (
        LOWEST5_BEFORE_AFTER_COMPARISON_PATH
    ),
    "all_nonhalide_exact_audit": (
        ALL_NONHALIDE_EXACT_AUDIT_PATH
    ),
    "exact_after_manifest": (
        LOWEST5_EXACT_AFTER_MANIFEST_PATH
    ),
    "experimental_pair_library": (
        EXPERIMENTAL_PAIR_LIBRARY_PATH
    ),
    "shared_ion_summary": (
        LOWEST5_SHARED_ION_SUMMARY_PATH
    ),
    "related_complete_IL_pairs": (
        LOWEST5_RELATED_COMPLETE_PAIRS_PATH
    ),
    "shared_experimental_records": (
        LOWEST5_SHARED_EXPERIMENTAL_RECORDS_PATH
    ),
    "condition_matched_records": (
        LOWEST5_CONDITION_MATCHED_RECORDS_PATH
    ),
    "best_coverage_pairs": (
        LOWEST5_BEST_COVERAGE_PAIRS_PATH
    ),
    "representative_comparison": (
        LOWEST5_REPRESENTATIVE_COMPARISON_PATH
    ),
    "evidence_scope_summary": (
        LOWEST5_EVIDENCE_SCOPE_SUMMARY_PATH
    ),
    "shared_ion_manifest": (
        LOWEST5_SHARED_ION_MANIFEST_PATH
    ),
    "deprecated_cleanup_audit": (
        DEPRECATED_CLEANUP_AUDIT_PATH
    ),
}

output_check_records = []

for (
    output_name,
    output_path,
) in required_output_paths.items():

    output_path = Path(
        output_path
    )

    output_check_records.append({
        "output_name": output_name,
        "file_name": output_path.name,
        "path": str(
            output_path
        ),
        "exists": bool(
            output_path.exists()
        ),
        "size_bytes": (
            int(
                output_path.stat().st_size
            )
            if output_path.exists()
            else 0
        ),
        "sha256": (
            sha256_file(
                output_path
            )
            if output_path.exists()
            else None
        ),
    })

output_check_df = pd.DataFrame(
    output_check_records
)

missing_outputs_df = (
    output_check_df.loc[
        ~output_check_df[
            "exists"
        ]
    ]
    .copy()
)

zero_size_outputs_df = (
    output_check_df.loc[
        output_check_df[
            "exists"
        ]
        & (
            output_check_df[
                "size_bytes"
            ]
            <= 0
        )
    ]
    .copy()
)

if not missing_outputs_df.empty:
    display(
        missing_outputs_df
    )

    raise RuntimeError(
        "仍有必要输出文件缺失。"
    )

if not zero_size_outputs_df.empty:
    display(
        zero_size_outputs_df
    )

    raise RuntimeError(
        "仍有必要输出文件大小为0。"
    )


# ============================================================
# 3. 候选正式口径审计
# ============================================================

if (
    len(
        lowest5_candidates_df
    )
    != EXPECTED_LOWEST5_COUNT
):
    raise RuntimeError(
        "正式最低5%候选数量不是854。"
    )

if not np.array_equal(
    lowest5_candidates_df[
        "rank_viscosity_nonhalide"
    ].to_numpy(
        dtype=int
    ),
    np.arange(
        1,
        EXPECTED_LOWEST5_COUNT + 1,
        dtype=int,
    ),
):
    raise RuntimeError(
        "最低5%候选排名不是1—854。"
    )

if lowest5_candidates_df[
    "candidate_pair_key_canonical"
].duplicated().any():
    raise RuntimeError(
        "最低5%候选存在重复完整离子对。"
    )


# ============================================================
# 4. 精确匹配审计
# ============================================================

if (
    lowest5_exact_before_df[
        "candidate_pair_key_canonical"
    ].nunique()
    != 0
):
    raise RuntimeError(
        "结构修正前的最低5%精确匹配数不是0。"
    )

if (
    lowest5_exact_after_df[
        "candidate_pair_key_canonical"
    ].nunique()
    != 0
):
    raise RuntimeError(
        "结构修正后的最低5%精确匹配数不是0。"
    )

if (
    all_nonhalide_exact_after_df[
        "candidate_pair_key_canonical"
    ].nunique()
    != 14
):
    raise RuntimeError(
        "全部17082种非简单卤素候选的"
        "精确匹配数不是14。"
    )

if (
    all_nonhalide_exact_after_df[
        "rank_viscosity_nonhalide"
    ].min()
    <= EXPECTED_LOWEST5_COUNT
):
    raise RuntimeError(
        "全候选精确匹配结果中出现了"
        "排名不大于854的候选。"
    )


# ============================================================
# 5. 共享离子证据审计
# ============================================================

if (
    len(
        candidate_evidence_summary_df
    )
    != EXPECTED_LOWEST5_COUNT
):
    raise RuntimeError(
        "共享离子证据汇总不是854行。"
    )

if candidate_evidence_summary_df[
    "candidate_pair_key_canonical"
].duplicated().any():
    raise RuntimeError(
        "共享离子证据汇总存在重复候选。"
    )

if not related_complete_IL_pairs_df.empty:
    shared_identity_check = (
        (
            related_complete_IL_pairs_df[
                "candidate_cation_smiles"
            ]
            == related_complete_IL_pairs_df[
                "experimental_cation_smiles"
            ]
        )
        |
        (
            related_complete_IL_pairs_df[
                "candidate_anion_smiles"
            ]
            == related_complete_IL_pairs_df[
                "experimental_anion_smiles"
            ]
        )
    )

    if not shared_identity_check.all():
        raise RuntimeError(
            "相关完整离子液体表中存在"
            "未共享规范化阳离子或阴离子的记录。"
        )

    exact_pair_check = (
        related_complete_IL_pairs_df[
            "candidate_pair_key_canonical"
        ]
        == related_complete_IL_pairs_df[
            "experimental_pair_key_canonical"
        ]
    )

    if exact_pair_check.any():
        raise RuntimeError(
            "共享离子相关表中混入了"
            "完整离子对精确匹配记录。"
        )

    if not (
        related_complete_IL_pairs_df[
            "shared_ion_count"
        ]
        == 1
    ).all():
        raise RuntimeError(
            "相关完整离子液体表中的"
            "共享离子数量不是1。"
        )

    allowed_relations = {
        "same_cation_different_anion",
        "different_cation_same_anion",
    }

    actual_relations = set(
        related_complete_IL_pairs_df[
            "shared_ion_relation"
        ].dropna()
    )

    if not actual_relations.issubset(
        allowed_relations
    ):
        raise RuntimeError(
            "存在未定义的共享离子关系。"
        )


# ============================================================
# 6. 温度与代表性记录审计
# ============================================================

if not condition_matched_records_df.empty:
    if not (
        condition_matched_records_df[
            "absolute_temperature_difference_to_candidate_K"
        ]
        <= TEMPERATURE_WINDOW_K
        + 1e-12
    ).all():
        raise RuntimeError(
            "温度窗口结果中存在"
            "超出298.15±0.50 K的记录。"
        )

if not representative_comparison_df.empty:
    if representative_comparison_df[
        "candidate_pair_key_canonical"
    ].duplicated().any():
        raise RuntimeError(
            "代表性对照表中同一候选出现多次。"
        )

    shared_candidate_count = int(
        (
            candidate_evidence_summary_df[
                "n_shared_complete_IL_pairs"
            ]
            > 0
        ).sum()
    )

    if (
        len(
            representative_comparison_df
        )
        != shared_candidate_count
    ):
        raise RuntimeError(
            "代表性对照表行数与具有共享离子"
            "证据的候选数量不一致。"
        )

    formal_candidate_key_set = set(
        lowest5_candidates_df[
            "candidate_pair_key_canonical"
        ]
    )

    representative_key_set = set(
        representative_comparison_df[
            "candidate_pair_key_canonical"
        ]
    )

    if not representative_key_set.issubset(
        formal_candidate_key_set
    ):
        raise RuntimeError(
            "代表性对照表包含854种正式候选之外的结构。"
        )


# ============================================================
# 7. 废弃输出清理审计
# ============================================================

remaining_deprecated_paths = [
    path
    for path in deprecated_paths
    if path.exists()
]

if remaining_deprecated_paths:
    raise RuntimeError(
        "仍存在已废弃的Top20、Top50或Top100输出：\n"
        + "\n".join(
            map(
                str,
                remaining_deprecated_paths,
            )
        )
    )


# ============================================================
# 8. 最终输出检查表
# ============================================================

FINAL_OUTPUT_CHECK_PATH = (
    CO2_OUTPUT_DIR
    / "lowest5pct_candidate_CO2_output_check.csv"
)

save_csv_atomic(
    output_check_df,
    FINAL_OUTPUT_CHECK_PATH,
)


# ============================================================
# 9. 最终运行清单
# ============================================================

FINAL_RUN_MANIFEST_PATH = (
    CO2_OUTPUT_DIR
    / (
        "lowest5pct_candidate_CO2_"
        "crosscheck_manifest.json"
    )
)

final_run_manifest = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "run_code_version": (
        CELL11_RUN_CODE_VERSION
    ),
    "status": (
        "lowest5pct_candidate_CO2_"
        "crosscheck_final_audit_completed"
    ),
    "formal_candidate_scope": {
        "description": (
            "lowest_5pct_non_simple_halide_candidates"
        ),
        "candidate_count": int(
            len(
                lowest5_candidates_df
            )
        ),
        "rank_min": int(
            lowest5_candidates_df[
                "rank_viscosity_nonhalide"
            ].min()
        ),
        "rank_max": int(
            lowest5_candidates_df[
                "rank_viscosity_nonhalide"
            ].max()
        ),
        "pred_eta_min_mPa_s": float(
            lowest5_candidates_df[
                "pred_eta_mPa_s"
            ].min()
        ),
        "pred_eta_max_mPa_s": float(
            lowest5_candidates_df[
                "pred_eta_mPa_s"
            ].max()
        ),
        "T_pred_K": TARGET_T_K,
        "P_pred_bar": TARGET_P_BAR,
    },
    "exact_complete_pair_results": {
        "before_structure_correction_candidates": int(
            lowest5_exact_before_df[
                "candidate_pair_key_canonical"
            ].nunique()
        ),
        "after_structure_correction_candidates": int(
            lowest5_exact_after_df[
                "candidate_pair_key_canonical"
            ].nunique()
        ),
        "after_structure_correction_records": int(
            len(
                lowest5_exact_after_df
            )
        ),
        "all_nonhalide_internal_audit_candidates": int(
            all_nonhalide_exact_after_df[
                "candidate_pair_key_canonical"
            ].nunique()
        ),
        "all_nonhalide_internal_audit_records": int(
            len(
                all_nonhalide_exact_after_df
            )
        ),
    },
    "shared_ion_results": (
        evidence_scope_summary_df.iloc[
            0
        ].to_dict()
    ),
    "comparison_policy": {
        "exact_complete_pair_basis": (
            "exact_canonical_cation_and_anion_identity"
        ),
        "related_complete_IL_basis": (
            "exact_canonical_cation_or_anion_identity"
        ),
        "molecular_fingerprint_used": False,
        "similarity_threshold_applied": False,
        "Henry_constant_calculated": False,
        "different_temperatures_combined": False,
        "different_pressures_combined": False,
        "solubility_values_averaged": False,
    },
    "verification": {
        "formal_candidate_count_is_854": True,
        "formal_candidate_ranks_are_1_to_854": True,
        "formal_candidate_pair_keys_unique": True,
        "formal_exact_match_count_is_zero": True,
        "all_nonhalide_exact_match_count_is_14": True,
        "all_related_pairs_share_exact_cation_or_anion": True,
        "related_pairs_exclude_exact_complete_pairs": True,
        "condition_records_within_temperature_window": True,
        "representative_candidate_keys_within_formal_scope": True,
        "original_CO2_database_modified": False,
        "corrected_CO2_database_modified": False,
        "deprecated_top_outputs_remaining": 0,
        "figure_catalog_for_word_created": False,
        "table_catalog_for_word_created": False,
    },
    "formal_outputs": {
        key: str(
            value
        )
        for key, value
        in required_output_paths.items()
    },
    "output_check": str(
        FINAL_OUTPUT_CHECK_PATH
    ),
    "deprecated_cleanup_audit": str(
        DEPRECATED_CLEANUP_AUDIT_PATH
    ),
    "output_root": str(
        CO2_OUTPUT_DIR
    ),
}

save_json_atomic(
    final_run_manifest,
    FINAL_RUN_MANIFEST_PATH,
)


# ============================================================
# 10. 输出最终审计结果
# ============================================================

print(
    "=" * 112
)

print(
    "Cell 11 最低5%候选与CO2溶解度数据库"
    "对照最终审计完成 ✓"
)

print(
    "=" * 112
)

print(
    "formal candidates                         =",
    len(
        lowest5_candidates_df
    ),
)

print(
    "formal exact complete-pair matches         =",
    lowest5_exact_after_df[
        "candidate_pair_key_canonical"
    ].nunique(),
)

print(
    "all nonhalide internal exact matches       =",
    all_nonhalide_exact_after_df[
        "candidate_pair_key_canonical"
    ].nunique(),
)

print(
    "candidates with any shared ion             =",
    int(
        (
            candidate_evidence_summary_df[
                "n_shared_complete_IL_pairs"
            ]
            > 0
        ).sum()
    ),
)

print(
    "related complete IL pair links             =",
    len(
        related_complete_IL_pairs_df
    ),
)

print(
    "condition-matched experimental records     =",
    len(
        condition_matched_records_df
    ),
)

print(
    "representative candidate comparisons       =",
    len(
        representative_comparison_df
    ),
)

print(
    "deprecated Top outputs remaining           =",
    len(
        remaining_deprecated_paths
    ),
)

print(
    "output check                               =",
    FINAL_OUTPUT_CHECK_PATH,
)

print(
    "final run manifest                         =",
    FINAL_RUN_MANIFEST_PATH,
)

print(
    "\nEvidence scope summary:"
)

display(
    evidence_scope_summary_df
)

print(
    "\nDeprecated-output cleanup audit:"
)

display(
    deprecated_cleanup_df
)

print(
    "\nFinal output check:"
)

display(
    output_check_df
)

Cell 11 最低5%候选与CO2溶解度数据库对照最终审计完成 ✓
formal candidates                         = 854
formal exact complete-pair matches         = 0
all nonhalide internal exact matches       = 14
candidates with any shared ion             = 427
related complete IL pair links             = 2138
condition-matched experimental records     = 9596
representative candidate comparisons       = 427
deprecated Top outputs remaining           = 0
output check                               = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\lowest5pct_candidate_CO2_output_check.csv
final run manifest                         = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\low_viscosity_CO2_solubility_crosscheck\lowest5pct_candidate_CO2_crosscheck_manifest.json

Evidence scope summary:


,scope,total_candidates,exact_complete_pair_candidates,both_ions_individually_observed_candidates,cation_only_observed_candidates,anion_only_observed_candidates,neither_ion_observed_candidates,candidates_with_any_shared_ion,related_candidate_experimental_pair_links,related_experimental_records,records_within_29815K_window,representative_candidate_comparisons
0,Lowest5pct_nonhalide,854,0,26,26,375,427,427,2138,145858,9596,427



Deprecated-output cleanup audit:


,file_name,path,existed_before_cleanup,exists_after_cleanup,removed_successfully
0,top100_candidate_shared_ion_match_all_pairs.csv,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,False,True
1,top100_candidate_shared_ion_summary.csv,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,False,True
2,top_candidate_shared_ion_scope_summary.csv,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,False,True
3,top100_candidate_shared_ion_best_coverage_pair...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,False,True
4,top100_candidate_shared_ion_experimental_recor...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,False,True
5,top100_candidate_shared_ion_evidence_manifest....,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,False,True
6,candidate_CO2_exact_match_counts_by_scope_afte...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,False,True



Final output check:


,output_name,file_name,path,exists,size_bytes,sha256
0,exact_records_before,lowest5pct_candidate_CO2_exact_pair_matched_re...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,621,cab62f7a772a22e073f5126195c6812b72a039a5963475...
1,exact_summary_before,lowest5pct_candidate_CO2_exact_pair_match_summ...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,326,38851d0fe9101a9440d219f4dd034767c2df447f318291...
2,exact_counts_before,lowest5pct_candidate_CO2_exact_match_counts_be...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,217,a0390545d23340e6fb7dc663f319d3532e285f40fcd9cf...
3,exact_T_window_before,lowest5pct_candidate_CO2_exact_match_T29815_wi...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,621,cab62f7a772a22e073f5126195c6812b72a039a5963475...
4,exact_before_manifest,lowest5pct_candidate_CO2_exact_match_before_co...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,2632,d6ed30b2c61489f7565fbcdff2cf7825787a79733aedb8...
5,exact_records_after,lowest5pct_candidate_CO2_exact_pair_matched_re...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,621,cab62f7a772a22e073f5126195c6812b72a039a5963475...
6,exact_summary_after,lowest5pct_candidate_CO2_exact_pair_match_summ...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,326,38851d0fe9101a9440d219f4dd034767c2df447f318291...
7,exact_counts_after,lowest5pct_candidate_CO2_exact_match_counts_af...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,267,4214be45f80b002352de175450ff4a95aef1833bba00d8...
8,exact_T_window_after,lowest5pct_candidate_CO2_exact_match_T29815_wi...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,621,cab62f7a772a22e073f5126195c6812b72a039a5963475...
9,before_after_comparison,lowest5pct_candidate_CO2_exact_match_before_af...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,310,4ef79fceb530e18e336d9b99f89c59666ed723388fa3c5...
